# Reproducing de Paula et al. (2026) from DANDI:000713

Paper. E. V. de Paula, R. M. Jungmann, A. J. Fontenele, L. A. A. Aguiar, P. V. Carelli, F. S. Matias, M. Copelli and N. A. P. de Vasconcelos, "Revealing Stimulus-Dependent Dynamics Through Statistical Complexity", PRX Life 4, 023018 (2026), doi:10.1103/wndc-k654.

Dataset. DANDI:000713, "Allen Institute - Visual Behavior - Neuropixels", version 0.240702.1725 (doi:10.48324/dandi.000713/0.240702.1725). This is the version the paper cites (its Ref. 22, as the claims file gives it), and it is the only published version of the dandiset; the draft has the same 4293 assets and was last modified on 2024-07-02, when this version was published. The 153 ecephys session files are streamed from the archive; nothing is written to disk.

Text. The claims file `claims/paula-2026.yaml` was written from the version of record (corrected 22 July 2026) and its Supplemental Material. The paper links no analysis code.

Claims. All 38 claims of the claims file are tested here; none is blocked. Claim `fig3a-reference-systems` uses synthetic series only (`data: elsewhere`, no recordings). The outcomes are in the attempt record and the summary at the end.

| Claim | Tested here |
|---|---|
| table1-sample | tested |
| fig1c-cv-region-medians | tested |
| fig1c-cv-above-surrogate | tested |
| fig1d-surrogate-abolishes-cv-correlations | tested |
| fig1e-positive-cv-modules | tested |
| fig1e-dg-ca3-anticorrelation-hubs | tested |
| fig1e-midbrain-weakest-cv-correlations | tested |
| fig2a-vis-cv-by-condition | tested |
| fig2a-thalamus-cv-by-condition | tested |
| fig2a-hippocampus-cv-stable | tested |
| fig2a-midbrain-cv-invariant | tested |
| fig2b-vis-summed-js-cv | tested |
| fig2b-hp-mb-summed-js-cv | tested |
| fig2b-visually-driven-js-cv-low | tested |
| fig2b-dg-ca3-high-js-cv | tested |
| fig2b-legend-js-cv | tested |
| fig3a-reference-systems | tested (synthetic series) |
| fig3b-complexity-levels-by-region | tested |
| fig3c-vis-ch-plane | tested |
| fig3d-th-ch-plane | tested |
| fig3e-hp-ch-plane | tested |
| fig3f-mb-ch-plane | tested |
| fig3-complexity-range-fractions | tested |
| fig4a-vis-complexity-by-condition | tested |
| fig4b-thalamus-lgd-lp-modulated | tested |
| fig4b-other-thalamus-not-modulated | tested |
| fig4c-dg-ca3-complexity | tested |
| fig4d-midbrain-complexity-invariant | tested |
| fig4e-js-complexity-natural-images | tested |
| fig4fg-js-complexity-active-blanks | tested |
| fig4h-js-complexity-spontaneous-converges | tested |
| supp-robustness-parameters | tested |
| fig5-vis-complexity-cv-coupling | tested |
| fig5-th-weaker-coupling | tested |
| fig5-hp-spontaneous-elevated | tested |
| fig5-mb-flat | tested |
| supp-fig6-mean-correlation | tested |
| complexity-separates-images-from-spontaneous | tested |

Method. The helper module `paula.py` reads each session and computes, for every local population (the units of one subregion in one session with at least 30 units) under each of three unit readings, and every 100-s window of the four conditions, the CV, the permutation entropy H and the statistical complexity C of the population rate, the same for an ISI-shuffled surrogate, the mean pairwise correlation (visual cortex and midbrain), and C and H for the parameter combinations of the supplement. The unit readings are 'vbn' (adopted; the filter of the Allen Institute's Visual Behavior Neuropixels tutorial, which matches Table I's neurons per population), and the claims file's 'all' and 'qc'. Its docstring says where each parameter comes from. Every claim's section below starts from that table and gives the criterion under each unit reading.

In [ ]:
import os
for v in ("OPENBLAS_NUM_THREADS", "OMP_NUM_THREADS", "MKL_NUM_THREADS"):
    os.environ[v] = "1"  # set before numpy loads: the four worker processes each run many small fits
import time, zlib, warnings
import numpy as np
import pandas as pd
from scipy import stats
import matplotlib.pyplot as plt
import paula

warnings.filterwarnings("ignore", category=RuntimeWarning)
pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 40)
pd.set_option("display.max_rows", 200)
print("dandiset", paula.DANDISET, "version", paula.VERSION)

## Loading

One pass over all ecephys sessions, four sessions at a time in separate processes. Each session's spike times are read up to the end of its spontaneous epoch, and every population is measured under the three unit readings ('vbn', 'all', 'qc'). This takes about 46 minutes.

In [ ]:
t0 = time.time()
sessions, n_assets = paula.list_sessions()
metas, windows, cv10_list, failed = paula.run_all(sessions, workers=4)
t_extract = time.time() - t0
print(f"{len(sessions)} session files of {n_assets} assets; extraction took {t_extract / 60:.1f} min")

In [ ]:
import resource
print("sessions read:", len(metas), "failed:", len(failed), failed)
print("subjects:", metas.subject.nunique(), "| sessions per subject:", metas.subject.value_counts().value_counts().to_dict())
print("session types:", metas.session_description.value_counts().to_dict())
print("genotypes:", metas.genotype.value_counts().to_dict())
print("probes with units per session:", metas.n_probes.value_counts().to_dict())
metas["image_set"] = np.where(metas.session_description.str.contains("_G_"), "G", "H")
metas["novel"] = metas.novel_fraction > 0
print("image set by novel images (is_image_novel):\n", pd.crosstab(metas.image_set, metas.novel))
print("no grouping of sessions gives 102: by genotype", metas.genotype.value_counts().to_dict(), "| by image set and novelty",
      metas.groupby(["image_set", "novel"]).size().to_dict(), "| six probes", int((metas.n_probes == 6).sum()))
print("units per session: median", metas.n_units.median(), "range", metas.n_units.min(), "to", metas.n_units.max())
n_all = sum(sum(d.values()) for d in metas.region_counts_all); n_good = sum(sum(d.values()) for d in metas.region_counts_good)
print(f"units with quality label 'good': {n_good} of {n_all} ({100 * n_good / n_all:.1f}%); the rest are labelled 'noise'")
print("active block length (s):", (metas.active_end - metas.active_start).describe().round(1).to_dict())
print("active windows per session:", metas.n_active_windows.value_counts().to_dict())
print("spontaneous epochs after the task (s):", pd.Series([e[-1][1] - e[-1][0] for e in metas.spont_epochs]).describe().round(1).to_dict())
print("images per session:", metas.n_images.value_counts().to_dict(), "| flashes shown per session: median", metas.n_flashes.median(),
      "| omitted: median", metas.n_omitted.median(), "| change flashes: median", metas.n_changes.median())
# blanks (Methods II.A gives about 240 s per image): flashes per image, and the gray-screen pieces after them
fpi = pd.Series([v for d in metas.flashes_per_image for v in d.values()])
cpi = pd.Series([v for d in metas.changes_per_image for v in d.values()])
print("shown flashes per image (all sessions):", fpi.describe().round(1).to_dict())
print("change flashes per image:", cpi.describe().round(1).to_dict())
print("blank pieces of 0.5 s:", int(metas.blank_piece_full.sum()), "| shorter (cut by the next onset):", int(metas.blank_piece_short.sum()),
      "| median gray screen after a shown flash (s):", round(metas.gray_after_flash_median.median(), 4))
print("blank series per image (s) = 0.5 s x flashes: median", round(0.5 * fpi.median(), 1), "| without change flashes", round(0.5 * (fpi - cpi).median(), 1),
      "| flashes per image for 240 s: 480")
sl = windows[windows.units == paula.MAIN_UNITS].groupby(["condition", "reading"]).series_length.describe()[["min", "50%", "max"]].round(1)
print("length of the series each window is cut from (s), units=vbn:\n", sl)
print("window rows by unit reading and condition:\n", windows.groupby(["units", "condition", "reading"]).size())
for col, unit in [("cv", ""), ("H", ""), ("C", ""), ("rate", "spikes/s per unit"), ("corr50", ""), ("corr20", "")]:
    x = windows[col]
    print(f"{col}: {np.nanmin(x):.3f} to {np.nanmax(x):.3f} {unit}, NaN {int(x.isna().sum())}")
# recordings deposited more than once: identical spike-time fingerprints or spike counts
fp = metas.fingerprint.apply(lambda v: tuple(np.round(v, 6)))
print("sessions sharing a spike-time fingerprint:", int(fp.duplicated(keep=False).sum()),
      "| sharing a total spike count:", int(metas.n_spikes.duplicated(keep=False).sum()),
      "| sharing subject and start time:", int(metas.duplicated(["subject", "session_start_time"], keep=False).sum()))
print("peak memory of this process (GB):", resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / 1e6)

### Clock check

The stimulus table and the spike times must share a clock. For every session the notebook counts the visual cortex spikes 30 to 130 ms after each flash onset minus those in the 100 ms before, for the true onsets and for onsets shifted by 375 ms into the gray screen, and for the omitted flashes (no image appears, so no response is expected).

In [ ]:
cl = metas[["clock_true", "clock_shift", "clock_omitted", "n_vis_units_clock"]]
print(cl.describe().round(2))
print("sessions where the response to true onsets exceeds that to shifted onsets:", int((cl.clock_true > cl.clock_shift).sum()), "of", len(cl))
print("sessions where the true-onset response exceeds the omitted-flash response:", int((cl.clock_true > cl.clock_omitted).sum()), "of", len(cl))
print("smallest true-onset contrast:", round(cl.clock_true.min(), 2), "| largest shifted contrast:", round(cl.clock_shift.max(), 2))

## table1-sample

> The analysis rests on 102 sessions and 1437 local populations (VIS 586, TH 313, HP 363, MB 175), with 53 985 visual cortex, 20 541 thalamic, 42 396 hippocampal and 17 163 midbrain neurons, and 77 216 analysis windows (11 321 natural images, 51 704 active behavior, 11 321 blanks, 2870 spontaneous).

The claims file names two unit readings, 'all' units and 'qc' (ISI violations < 0.5, amplitude cutoff < 0.1, presence ratio > 0.9), and says that Table I's neuron counts decide between them. The dandiset has 153 sessions against the paper's 102, so totals cannot be compared as they stand: the comparison that can tell filters apart is per population (neurons per population) or after scaling the totals by 102/153. The first table gives that comparison for six filters, from the unit counts of every session: the claims file's two, 'vbn' (snr > 1, ISI violations < 1, firing rate > 0.1 spikes/s, the unit filter of the Allen Institute's Visual Behavior Neuropixels quickstart tutorial, AllenSDK commit a56bd8ea), 'good' (quality label 'good'), 'qc95' ('qc' with presence ratio > 0.95) and 'good_pr95'. The second gives the claim's counts from the windows for the three readings run below, under each session reading ((a) sessions with at least one kept population; (b) those that also have the spontaneous gray-screen epoch).

In [ ]:
TABLE1 = {"VISal": 8572, "VISam": 8566, "VISl": 9255, "VISp": 10010, "VISpm": 10250, "VISrl": 7332,
          "LGd": 2169, "LP": 2795, "MGd": 2497, "MGm": 1928, "MGv": 4492, "PIL": 868, "POL": 1298, "PoT": 1461, "SGN": 3033,
          "CA1": 22399, "CA3": 3729, "DG": 7486, "POST": 1367, "ProS": 3067, "SUB": 4348,
          "APN": 10716, "NB": 793, "NOT": 309, "MRN": 3661, "SCig": 1684}
EXP1 = {"sessions": 102, "VIS populations": 586, "TH populations": 313, "HP populations": 363, "MB populations": 175,
        "VIS neurons": 53985, "TH neurons": 20541, "HP neurons": 42396, "MB neurons": 17163,
        "natural image windows": 11321, "active behavior windows": 51704, "blank windows": 11321, "spontaneous windows": 2870}
has_spont = set(metas.session_index[metas.n_spont_windows >= 2])
SCALE = 102 / len(metas)
# 1. every filter, from the unit counts of each session: populations (>= 30 units of one Table I subregion in a session,
#    the subregion kept when it has such populations in >= 8 sessions) and neurons, as counted and scaled to 102 sessions
flt = []
for f in paula.COUNT_FILTERS:
    rows = [(m.session_index, r, c) for m in metas.itertuples() for r, c in getattr(m, f"region_counts_{f}").items()]
    rc = pd.DataFrame(rows, columns=["session", "region", "n"])
    rc = rc[rc.n >= paula.MIN_UNITS]
    ns = rc.groupby("region").session.nunique()
    lab = sorted(ns[ns >= paula.MIN_SESSIONS].index)
    print(f"filter {f}: labels kept by the rule {len(lab)}; not in Table I {[l for l in lab if l not in paula.REGION_AREA]}; "
          f"Table I subregions not kept {[r for r in paula.REGION_ORDER if r not in lab]}")
    p = rc[rc.region.isin([l for l in lab if l in paula.REGION_AREA])].copy()
    p["area"] = p.region.map(paula.REGION_AREA)
    per_reg = p.groupby("region").n.sum().reindex(paula.REGION_ORDER) * SCALE / pd.Series(TABLE1)
    for a in paula.AREAS:
        x = p[p.area == a]
        flt.append({"filter": f, "area": a, "populations": len(x), "populations x 102/153": round(len(x) * SCALE),
                    "Table I populations": EXP1[f"{a} populations"], "neurons": int(x.n.sum()), "neurons x 102/153": round(x.n.sum() * SCALE),
                    "Table I neurons": EXP1[f"{a} neurons"], "scaled neuron ratio": round(x.n.sum() * SCALE / EXP1[f"{a} neurons"], 2),
                    "neurons per population": round(x.n.mean(), 1), "Table I per population": round(EXP1[f"{a} neurons"] / EXP1[f"{a} populations"], 1),
                    "subregion ratios (scaled)": f"{per_reg[[r for r in paula.AREAS[a]]].min():.2f}-{per_reg[[r for r in paula.AREAS[a]]].max():.2f}"})
flt = pd.DataFrame(flt)
print(flt.to_string())
fsum = flt.assign(err=lambda d: np.abs(np.log(d["scaled neuron ratio"]))).groupby("filter").err.max()
print("largest |log ratio| of the scaled area neuron counts to Table I, by filter:", fsum.round(3).to_dict())
# 2. the three unit readings run below: counts from the windows themselves, under both session readings
t1 = {}
for ur in paula.UNIT_READINGS:
    regs = paula.kept_regions(windows, ur)
    for sr in ["a", "b"]:
        w = paula.conditions(windows, ur, "a", "a")
        w = w[w.region.isin(regs)]
        if sr == "b":
            w = w[w.session.isin(has_spont)]
        pops = w[w.condition == "active"].drop_duplicates(["session", "region"])
        res = {"sessions": pops.session.nunique()}
        for a in paula.AREAS:
            res[f"{a} populations"] = int((pops.area == a).sum())
        for a in paula.AREAS:
            res[f"{a} neurons"] = int(pops.loc[pops.area == a, "n"].sum())
        res["natural image windows"] = int((w.condition == "images").sum())
        res["active behavior windows"] = int((w.condition == "active").sum())
        res["blank windows"] = int((w.condition == "blanks").sum())
        res["spontaneous windows"] = int((w.condition == "spont").sum())
        t1[f"{ur}/{sr}"] = res
        if sr == "a":
            per = pops.groupby("region").agg(populations=("n", "size"), neurons=("n", "sum")).reindex(paula.REGION_ORDER)
            per["Table I neurons"] = pd.Series(TABLE1)
            per["ratio"] = (per.neurons / per["Table I neurons"]).round(2)
            per["ratio x 102/153"] = (per.neurons * SCALE / per["Table I neurons"]).round(2)
            print(f"\nunits={ur}: per subregion (kept regions: {len(regs)})\n", per.T.to_string())
tab = pd.DataFrame(t1)
tab["expected"] = pd.Series(EXP1)
tab["tolerance"] = tab["expected"] * 0.1
for c in t1:
    tab["within " + c] = (tab[c] - tab["expected"]).abs() <= tab["tolerance"]
print(tab.to_string())
for c in t1:
    print(c, "all within tolerance:", bool(tab["within " + c].all()), "| session and population counts within:",
          bool(tab.loc[["sessions"] + [f"{a} populations" for a in paula.AREAS], "within " + c].all()))
print("blank windows under reading b:", int((paula.conditions(windows, paula.MAIN_UNITS, "b", "b").condition == "blanks").sum()))

**Result: not reproduced.** Every one of the 153 ecephys sessions has kept populations and the spontaneous epoch, so both session readings give 153 sessions, not 102, under every unit reading, and no reading puts the counts within tolerance. Scaled to 102 sessions, the 'vbn' filter gives area neuron counts 1.00 (VIS), 1.02 (TH), 0.93 (HP) and 0.94 (MB) times Table I's, per-subregion ratios 0.79 to 1.21, and neurons per population 92.7, 64.7, 112.0 and 98.0 against Table I's 92.1, 65.6, 116.8 and 98.1; it keeps all 26 subregions. 'all' gives 1.28 to 1.43 times Table I and 'qc' 0.34 to 0.55 times (qc also drops NOT); 'qc' puts the four window counts within tolerance only because its fewer populations per session (754 VIS populations, 503 when scaled to 102 sessions, against 586) offset the excess of sessions. 'vbn' is adopted by the other claims as an assumption, the filter closest to Table I by the comparison the claims file names, with 'all' and 'qc' run as variants for every claim.

### Readings used below

The adopted unit reading is 'vbn' (assumption, above); every claim is also run under 'all' and 'qc', and each claim cell prints the criterion's verdict under each unit reading. The surrogate is drawn under five seeds for 'vbn' only. The four condition readings combine the blank reading (a: after every shown flash; b: only after flashes followed by another flash) with the spontaneous reading (a: the first two 100-s windows of the gray screen; b: two windows centred in it). They are written `blanks/spont`, for example `a/b`.

In [ ]:
UNITS = paula.UNIT_READINGS
MAIN = paula.MAIN_UNITS
CR = [("a", "a"), ("a", "b"), ("b", "a"), ("b", "b")]
SFX = ["", "_s1000", "_s2000", "_s3000", "_s4000"]
VD = ["images", "active", "blanks"]
REGS, WD, CACHE, OUT = {}, {}, {}, {}
for ur in UNITS:
    REGS[ur] = paula.kept_regions(windows, ur)
    WD[ur] = {}
    for b, s in CR:
        x = paula.conditions(windows, ur, b, s)
        WD[ur][f"{b}/{s}"] = x[x.region.isin(REGS[ur])]
    CACHE[ur] = {}
    a = WD[ur]["a/a"]
    print(f"units={ur}: {len(REGS[ur])} subregions; {a[a.condition == 'active'].drop_duplicates(['session', 'region']).shape[0]} populations; "
          f"active windows {int((a.condition == 'active').sum())}; not kept {[r for r in paula.REGION_ORDER if r not in REGS[ur]]}")

def use(ur):
    """Make `ur` the current unit reading: W, ACT, IMG, REG and the per-reading tables computed so far."""
    g = globals()
    g.update(UR=ur, W=WD[ur], REG=REGS[ur], SFXU=SFX if ur == MAIN else [""])
    g.update(ACT=W["a/a"][W["a/a"].condition == "active"], IMG=W["a/a"][W["a/a"].condition == "images"])
    g.update(VISR=[r for r in REG if paula.REGION_AREA[r] == "VIS"], THR=[r for r in REG if paula.REGION_AREA[r] == "TH"],
             HPR=[r for r in REG if paula.REGION_AREA[r] == "HP"], MBR=[r for r in REG if paula.REGION_AREA[r] == "MB"])
    g.update(iu=np.triu_indices(len(REG), 1), AREA_OF=np.array([paula.REGION_AREA[r] for r in REG]))
    g.update(CACHE[ur])
    print(f"--- units={ur}" + (" (adopted)" if ur == MAIN else " (variant)"))

def verdict(claim, ur, v):
    OUT.setdefault(claim, {})[ur] = v

def within(x, lo, hi):
    return (x >= lo - 1e-12) & (x <= hi + 1e-12)

def widen(lo, hi):
    return lo - 0.1 * abs(lo), hi + 0.1 * abs(hi)

def tol(x, v, t):
    return abs(x - v) <= t + 1e-12

def REG_OF(d):
    return [r for r in paula.REGION_ORDER if r in set(d.region)]

## fig1c-cv-region-medians

> During active behavior visual cortex subregions have the highest population CV (medians 0.40 to 0.59), thalamic subregions moderate (0.27 to 0.37), hippocampal subregions high (0.25 to 0.49, dentate gyrus highest at 0.49) and midbrain the lowest (0.16 to 0.45).

In [ ]:
RANGES_CV = {"VIS": (0.40, 0.59), "TH": (0.27, 0.37), "HP": (0.25, 0.49), "MB": (0.16, 0.45)}
IQR_CV = {"VIS": (0.12, 0.18), "TH": (0.10, 0.15), "HP": (0.08, 0.13), "MB": (0.05, 0.23)}

def cv_medians(act):
    g = act.groupby("region").cv
    t = pd.DataFrame({"median": g.median(), "iqr": g.quantile(0.75) - g.quantile(0.25), "windows": g.size()}).reindex(REG_OF(act))
    t["area"] = [paula.REGION_AREA[r] for r in t.index]
    return t

def judge_fig1c(t, label):
    ok_all, ok_iqr = True, True
    for a, (lo, hi) in RANGES_CV.items():
        x = t[t.area == a]
        wl, wh = widen(lo, hi)
        inside = within(x["median"], wl, wh)
        ext = tol(x["median"].min(), lo, 0.1 * lo) and tol(x["median"].max(), hi, 0.1 * hi)
        il, ih = widen(*IQR_CV[a])
        iqr_in = within(x["iqr"], il, ih)
        print(f"{label} {a}: medians {x['median'].min():.3f} to {x['median'].max():.3f} (expected {lo} to {hi}); inside widened range {int(inside.sum())}/{len(x)}; "
              f"extremes within tolerance {ext}; IQR {x['iqr'].min():.3f} to {x['iqr'].max():.3f}, inside {int(iqr_in.sum())}/{len(x)}")
        ok_all &= bool(inside.all()) and ext
        ok_iqr &= bool(iqr_in.all())
    hp = t[t.area == "HP"]["median"]
    print(f"{label}: highest hippocampal median is {hp.idxmax()} ({hp.max():.3f}); medians and extremes all match: {ok_all}; IQRs all match: {ok_iqr}")
    return ok_all and hp.idxmax() == "DG", ok_iqr

fam = set(metas.session_index[metas.image_set == "G"])
for ur in UNITS:
    use(ur)
    t = cv_medians(ACT)
    print(t.round(3).T.to_string())
    ok_m, ok_i = judge_fig1c(t, f"units={ur}")
    verdict("fig1c-cv-region-medians", ur, "reproduced" if ok_m and ok_i else "partially_reproduced" if ok_m else "not_reproduced")
    # by session group: the paper's sessions are not identified, so the result is also given for familiar and novel image sets
    for name, ss in [("image set G", fam), ("image set H", set(metas.session_index) - fam)]:
        x = ACT[ACT.session.isin(ss)].groupby("area").cv.median()
        print(name, "area medians:", x.round(3).to_dict())
print(OUT["fig1c-cv-region-medians"])

**Result: partially reproduced.** Under 'vbn' every subregion median lies in its area's widened range and every extreme is within tolerance (VIS 0.416 to 0.615, TH 0.281 to 0.384, HP 0.256 to 0.491 with DG highest, MB 0.165 to 0.469), but the IQRs miss in 3 of 6 visual subregions (0.150 to 0.213 against 0.108 to 0.198) and 2 of 9 thalamic ones, which the criterion makes partially reproduced. Under 'all' and 'qc' the medians miss (DG 0.564 and NB 0.564 under 'all', LP 0.414 under 'qc'). The outcome rests on the adopted unit reading ('vbn'); the table at the end gives the criterion under 'all' and 'qc'.

## fig1c-cv-above-surrogate

> Empirical CV exceeds that of ISI-shuffled surrogates in every brain region (VIS 2 to 3 times, surrogate medians 0.21 to 0.24; TH 1.3 to 1.6; HP 1.5 to 2.2; MB 1.3 to 1.9), with Mann-Whitney p < 0.001 for all regions.

One surrogate per active window (Methods II.E). The surrogate is drawn under the seed of this claim, `zlib.crc32(b"paula-2026/fig1c-cv-above-surrogate")`, and, for the adopted unit reading, under that seed plus 1000, 2000, 3000 and 4000; the same surrogates serve the claims of Figs. 1d and 3 (see the deviations).

In [ ]:
RATIO = {"VIS": (2, 3), "TH": (1.3, 1.6), "HP": (1.5, 2.2), "MB": (1.3, 1.9)}

def surrogate_check(act, sfx, verbose):
    g = act.groupby("region")
    t = pd.DataFrame({"emp": g.cv.median(), "surr": g["s_cv" + sfx].median()}).reindex(REG_OF(act))
    t["ratio"] = t.emp / t.surr
    t["area"] = [paula.REGION_AREA[r] for r in t.index]
    p_sub = {r: stats.mannwhitneyu(x.cv.dropna(), x["s_cv" + sfx].dropna()).pvalue for r, x in g}
    t["p"] = pd.Series(p_sub)
    ok_ratio = True
    for a, (lo, hi) in RATIO.items():
        wl, wh = widen(lo, hi)
        x = t[t.area == a]
        ok_ratio &= bool(within(x.ratio, wl, wh).all())
        if verbose:
            print(f"{a}: ratios {x.ratio.min():.2f} to {x.ratio.max():.2f} (expected {lo} to {hi}, widened {wl:.2f} to {wh:.2f}); inside {int(within(x.ratio, wl, wh).sum())}/{len(x)}")
    vs = t[t.area == "VIS"].surr
    ok_vs = bool(within(vs, 0.189, 0.264).all())
    pa = {}
    for a, x in act.groupby("area"):
        pa[a] = (stats.mannwhitneyu(x.cv.dropna(), x["s_cv" + sfx].dropna()).pvalue, x.cv.median() > x["s_cv" + sfx].median())
    ok_sub = bool((t.p < 0.001).all() and (t.emp > t.surr).all())
    ok_area = all(p < 0.001 and hi for p, hi in pa.values())
    if verbose:
        print(t.round(4).T.to_string())
        print(f"VIS surrogate medians {vs.min():.3f} to {vs.max():.3f} (0.189 to 0.264): {ok_vs}")
        print("per area Mann-Whitney:", {a: (f"{p:.1e}", bool(h)) for a, (p, h) in pa.items()})
        print(f"tests hold per subregion: {ok_sub}; per area: {ok_area}; all ratios inside: {ok_ratio}; largest subregion p {t.p.max():.1e}")
        # windows are not independent: the same comparison on one value per population (its median over windows), Wilcoxon signed-rank
        pm = act.groupby(["area", "session", "region"])[["cv", "s_cv" + sfx]].median()
        print("per-population medians, Wilcoxon signed-rank by area:",
              {a: f"{stats.wilcoxon(x.cv, x['s_cv' + sfx]).pvalue:.1e} (n={len(x)}, empirical higher in {int((x.cv > x['s_cv' + sfx]).sum())})" for a, x in pm.groupby(level=0)})
    return t, ok_ratio, ok_vs, ok_sub, ok_area

for ur in UNITS:
    use(ur)
    t_, r_, v_, ps_, pa_ = surrogate_check(ACT, "", True)
    verdict("fig1c-cv-above-surrogate", ur, "reproduced" if r_ and v_ and (ps_ or pa_) else "partially_reproduced" if (ps_ or pa_) else "not_reproduced")
    if ur == MAIN:
        print("\nfive seeds (claim seed, +1000, +2000, +3000, +4000):")
        for s in SFX:
            t_, r_, v_, ps_, pa_ = surrogate_check(ACT, s, False)
            print(f"seed{s or ' (claim)'}: ratio range {t_.ratio.min():.3f} to {t_.ratio.max():.3f}; VIS surrogate medians {t_[t_.area=='VIS'].surr.min():.4f} to {t_[t_.area=='VIS'].surr.max():.4f}; "
                  f"ratios inside {r_}; VIS surrogates inside {v_}; tests per subregion {ps_}, per area {pa_}; largest p {t_.p.max():.1e}; "
                  f"HP ratios {t_[t_.area=='HP'].ratio.round(2).to_dict()}")
print(OUT["fig1c-cv-above-surrogate"])

**Result: partially reproduced.** Empirical CV is above the surrogate CV in every subregion and every area (Mann-Whitney p at most 1.4e-51 per subregion, 0 at printed precision per area; on one median per population, Wilcoxon p at most 3.3e-42, empirical higher in 246 of 247 to 875 of 875 populations), and the visual surrogate medians (0.233 to 0.250) lie in 0.189 to 0.264. Three ratios fall outside their widened ranges: VISam (1.75) and VISrl (1.78) below 1.80, and POST (1.29) below 1.35. All five seeds give the same result (ratios 1.286 to 2.644). 'all' and 'qc' give the same outcome. The outcome rests on the adopted unit reading ('vbn'); the table at the end gives the criterion under 'all' and 'qc'.

## fig1d-surrogate-abolishes-cv-correlations

> Inter-regional correlations of CV time series during active behavior are present in the data and abolished in the ISI-shuffled surrogates.

Per pair of subregions recorded in the same session, the Pearson correlation of their 10-s CV series over the active windows (360 values per session at most). Reading (a) concatenates the series of all shared sessions; reading (b) averages the per-session correlations.

In [ ]:
CVS = pd.DataFrame(cv10_list).sort_values(["session", "region", "k"])
CORR = {}

def corr_mats(series, by_sess, which):
    n = len(REG)
    ca, cb, ns = (np.full((n, n), np.nan) for _ in range(3))
    for i in range(n):
        for j in range(i + 1, n):
            xs, ys, rs = [], [], []
            for s, rr in by_sess.items():
                if REG[i] in rr and REG[j] in rr:
                    x, y = series[(s, REG[i])][which], series[(s, REG[j])][which]
                    ok = np.isfinite(x) & np.isfinite(y)
                    if ok.sum() > 2 and x[ok].std() > 0 and y[ok].std() > 0:
                        xs.append(x[ok]); ys.append(y[ok]); rs.append(np.corrcoef(x[ok], y[ok])[0, 1])
            if xs:
                ca[i, j] = ca[j, i] = np.corrcoef(np.concatenate(xs), np.concatenate(ys))[0, 1]
                cb[i, j] = cb[j, i] = np.mean(rs)
                ns[i, j] = ns[j, i] = len(rs)
    f = lambda m: pd.DataFrame(m, index=REG, columns=REG)
    return f(ca), f(cb), f(ns)

for ur in UNITS:
    use(ur)
    cvs = CVS[(CVS.units == ur) & CVS.region.isin(REG)]
    scols = ["s_cv10" + x for x in SFXU]
    series = {}
    for (s, r), g in cvs.groupby(["session", "region"]):
        series[(s, r)] = [np.concatenate(list(g.cv10))] + [np.concatenate(list(g[c])) for c in scols]
    by_sess = {}
    for (s, r) in series:
        by_sess.setdefault(s, []).append(r)
    EMP_A, EMP_B, NSH = corr_mats(series, by_sess, 0)
    SUR_A, SUR_B, _ = corr_mats(series, by_sess, 1)
    CORR[ur] = {"EMP_A": EMP_A, "EMP_B": EMP_B, "SUR_A": SUR_A, "SUR_B": SUR_B}
    print("pairs with at least one shared session:", int(np.isfinite(EMP_A.values[iu]).sum()), "of", len(iu[0]),
          "| shared sessions per pair: median", np.nanmedian(NSH.values[iu]))
    ok1d = {}
    for lab, e, s in [("a", EMP_A, SUR_A), ("b", EMP_B, SUR_B)]:
        ev, sv = e.values[iu], s.values[iu]
        ev, sv = ev[np.isfinite(ev)], sv[np.isfinite(sv)]
        q = np.percentile(np.abs(sv), [50, 90, 99, 100])
        ok1d[lab] = (ev.max() >= 0.45) and (np.abs(sv).max() < 0.2)
        print(f"reading {lab}: empirical r {ev.min():.3f} to {ev.max():.3f}, pairs >= 0.45: {int((ev >= 0.45).sum())}; "
              f"surrogate |r| median {q[0]:.3f}, 90th {q[1]:.3f}, 99th {q[2]:.3f}, max {q[3]:.3f}; pairs with |r| > 0.2: {int((np.abs(sv) > 0.2).sum())}; passes {ok1d[lab]}")
    verdict("fig1d-surrogate-abolishes-cv-correlations", ur, "reproduced" if all(ok1d.values()) else "not_reproduced")
    if ur == MAIN:
        fig, ax = plt.subplots(1, 2, figsize=(13, 5.5))
        for a_, (lab, e, s) in zip(ax, [("a", EMP_A, SUR_A), ("b", EMP_B, SUR_B)]):
            m = np.tril(e.values, -1) + np.triu(s.values, 1)
            im = a_.imshow(m, vmin=-0.7, vmax=0.7, cmap="RdBu_r")
            a_.set_xticks(range(len(REG)), REG, rotation=90, fontsize=7); a_.set_yticks(range(len(REG)), REG, fontsize=7)
            a_.set_title(f"reading {lab}: lower empirical, upper surrogate")
        plt.colorbar(im, ax=ax); plt.show()
        print("colour scale -0.7 to 0.7; values outside it:", int((np.abs(np.r_[EMP_A.values[iu], EMP_B.values[iu], SUR_A.values[iu], SUR_B.values[iu]]) > 0.7).sum()))
        for k, sfx in enumerate(SFX[1:], start=2):
            sa, sb, _ = corr_mats(series, by_sess, k)
            va, vb = np.abs(sa.values[iu]), np.abs(sb.values[iu])
            print(f"surrogate seed{sfx}: max |r| reading a {np.nanmax(va):.3f} (pairs > 0.2: {int((va > 0.2).sum())}), reading b {np.nanmax(vb):.3f} (pairs > 0.2: {int((vb > 0.2).sum())})")
print(OUT["fig1d-surrogate-abolishes-cv-correlations"])

**Result: not reproduced.** The empirical correlations are present (largest 0.798 under reading a and 0.679 under b), but the surrogate correlations are not abolished: 147 of 325 pairs (reading a) and 240 (reading b) have |r| above 0.2, largest 0.890 and 0.708, and the other four seeds give the same (0.889 to 0.900 and 0.699 to 0.708). The per-window ISI shuffle keeps each 100-s window's spike counts, so the surrogate CV still follows the rate of each window, and those rates co-vary across regions. Every unit reading gives the same outcome.

## fig1e-positive-cv-modules

> The strongest 10% of positive CV correlations (0.50 to 0.67) link subregions within functional modules (visual areas together, subicular regions together, thalamic populations together), with connections between distinct systems largely absent.

In [ ]:
def pairs_table(m):
    rows = [(REG[i], REG[j], m.values[i, j]) for i, j in zip(*iu) if np.isfinite(m.values[i, j])]
    p = pd.DataFrame(rows, columns=["r1", "r2", "r"])
    p["same_area"] = [paula.REGION_AREA[a] == paula.REGION_AREA[b] for a, b in zip(p.r1, p.r2)]
    return p

for ur in UNITS:
    use(ur)
    ok1e = {}
    for lab in ["a", "b"]:
        p = pairs_table(CORR[ur]["EMP_" + lab.upper()])
        top = p.nlargest(int(round(0.1 * len(p))), "r")
        lo_ok, hi_ok = tol(top.r.min(), 0.50, 0.05), tol(top.r.max(), 0.67, 0.067)
        frac = top.same_area.mean()
        ok1e[lab] = lo_ok and hi_ok and frac > 0.5
        print(f"reading {lab}: top {len(top)} of {len(p)} pairs, r {top.r.min():.4f} to {top.r.max():.4f} (expected 0.50 to 0.67; low end within tolerance {lo_ok}, high end {hi_ok}); "
              f"same major area {int(top.same_area.sum())}/{len(top)} ({frac:.2f}); passes {ok1e[lab]}")
        print("   ", "; ".join(f"{a}-{b} {r:.2f}" for a, b, r in top[["r1", "r2", "r"]].values))
    verdict("fig1e-positive-cv-modules", ur, "reproduced" if any(ok1e.values()) else "not_reproduced")
print(OUT["fig1e-positive-cv-modules"])

**Result: reproduced.** Under reading b (the per-session correlations averaged) the strongest 32 of 325 pairs have r 0.478 to 0.680, inside 0.50 +/- 0.05 to 0.67 +/- 0.067, and 28 of them join subregions of the same major area (visual areas with each other, thalamic nuclei, subicular pairs ProS-SUB and POST-SUB). Under reading a (concatenated) the top value is 0.798, above 0.737, with 21 of 32 within an area. Reading b is adopted as an assumption: concatenating the series of different sessions mixes session-level differences into one correlation, which averaging per session avoids. Under 'all' and 'qc' neither reading passes (for example reading b low end 0.450 under 'all'). The outcome rests on the adopted unit reading ('vbn'); the table at the end gives the criterion under 'all' and 'qc'.

## fig1e-dg-ca3-anticorrelation-hubs

> Among the 10% most anticorrelated region pairs (r from -0.27 to -0.002), DG and CA3 are the hubs of negative CV relationships with other regions.

In [ ]:
for ur in UNITS:
    use(ur)
    ok1h = {}
    for lab in ["a", "b"]:
        p = pairs_table(CORR[ur]["EMP_" + lab.upper()])
        neg = p.nsmallest(int(round(0.1 * len(p))), "r")
        cnt = pd.Series(list(neg.r1) + list(neg.r2)).value_counts()
        top2 = set(cnt.index[:2])
        tie = cnt.iloc[1] == cnt.iloc[2] if len(cnt) > 2 else False
        ok1h[lab] = top2 == {"DG", "CA3"} and tol(neg.r.min(), -0.27, 0.027)
        print(f"reading {lab}: {len(neg)} pairs, r {neg.r.min():.3f} to {neg.r.max():.3f} (most negative expected -0.27 +/- 0.027); "
              f"appearances {cnt.head(6).to_dict()}; tie for second place {tie}; passes {ok1h[lab]}")
    verdict("fig1e-dg-ca3-anticorrelation-hubs", ur, "reproduced" if any(ok1h.values()) else "not_reproduced")
print(OUT["fig1e-dg-ca3-anticorrelation-hubs"])

**Result: reproduced.** Under reading b, DG (16) and CA3 (14) are the subregions in most of the 32 most negative pairs, and the most negative value, -0.247, is within -0.27 +/- 0.027. Under reading a, CA3 and NB lead (8 each) and the most negative value is -0.436. Reading b is adopted, as for fig1e-positive-cv-modules. Under 'all' the most negative value is -0.178 and under 'qc' -0.359, both outside. The outcome rests on the adopted unit reading ('vbn'); the table at the end gives the criterion under 'all' and 'qc'.

## fig1e-midbrain-weakest-cv-correlations

> Midbrain areas have the weakest CV correlations with all other regions, near zero (-0.04 to +0.03).

In [ ]:
lo_n, hi_n = widen(-0.04, 0.03)
for ur in UNITS:
    use(ur)
    ok1m = {}
    for lab in ["a", "b"]:
        m = CORR[ur]["EMP_" + lab.upper()]
        mm = m.abs().mean(axis=1)
        area_mean = mm.groupby([paula.REGION_AREA[r] for r in mm.index]).mean()
        p = pairs_table(m)
        mbp = p[[paula.REGION_AREA[a] == "MB" or paula.REGION_AREA[b] == "MB" for a, b in zip(p.r1, p.r2)]]
        near = within(mbp.r, lo_n, hi_n).mean()
        ok1m[lab] = area_mean.idxmin() == "MB" and near > 0.5
        print(f"reading {lab}: mean |r| by area {area_mean.round(3).to_dict()}; per MB subregion {mm[MBR].round(3).to_dict()}; "
              f"MB pairs in {lo_n:.3f} to {hi_n:.3f}: {int(within(mbp.r, lo_n, hi_n).sum())}/{len(mbp)} ({near:.2f}); passes {ok1m[lab]}")
    verdict("fig1e-midbrain-weakest-cv-correlations", ur, "reproduced" if any(ok1m.values()) else "not_reproduced")
print(OUT["fig1e-midbrain-weakest-cv-correlations"])

**Result: not reproduced.** The midbrain has the lowest mean |r| under both readings (reading a 0.183 against HP 0.187, TH 0.253, VIS 0.325; reading b 0.210 against 0.252 to 0.360), but its correlations are not near zero: 13 of 115 (reading a) and 3 of 115 (reading b) midbrain pairs lie in -0.044 to 0.033. Every unit reading gives the same outcome.

## Medians of CV by condition (claims of Fig. 2a)

The median CV of each subregion in each condition, under each condition reading. Rows are subregions; columns condition and reading.

In [ ]:
def med_table(value, Wd):
    out = {}
    for k, w in Wd.items():
        out[k] = w.groupby(["region", "condition"])[value].median().unstack()[paula.CONDITIONS]
    return pd.concat(out, axis=1).reindex([r for r in paula.REGION_ORDER if r in set(Wd["a/a"].region)])

for ur in UNITS:
    CACHE[ur]["MED_CV"] = med_table("cv", WD[ur])
    CACHE[ur]["MED_C"] = med_table("C", WD[ur])
print("median CV, units=vbn:\n", CACHE[MAIN]["MED_CV"].round(3).to_string())
for ur in UNITS:
    print(f"windows per condition (units={ur}, reading a/a):", WD[ur]["a/a"].condition.value_counts().to_dict())

def check_expected(med, exp, readings=None):
    """exp: list of (label, region, condition(s), kind, value, tolerance); returns per reading the passes."""
    res = {}
    for rd in (readings or list(med.columns.levels[0])):
        rows = []
        for lab, reg, conds, kind, v, t in exp:
            if reg not in med.index:
                rows.append((lab, "not kept", v, False))
                continue
            vals = med.loc[reg, rd][conds] if isinstance(conds, list) else pd.Series({conds: med.loc[reg, (rd, conds)]})
            if kind == "each":
                ok = bool(all(tol(y, v, t) for y in vals))
                x = f"{vals.min():.3f}-{vals.max():.3f}"
            else:
                x = {"min": vals.min(), "max": vals.max(), "value": vals.iloc[0]}[kind]
                ok = tol(x, v, t)
                x = round(float(x), 3)
            rows.append((lab, x, v, ok))
        res[rd] = rows
    return res

def show_expected(res):
    for rd, rows in res.items():
        n = sum(r[3] for r in rows)
        print(f"reading {rd}: {n}/{len(rows)} within tolerance; " + "; ".join(f"{l} {x} ({v})" + ("" if ok else " MISS") for l, x, v, ok in rows))
    return {rd: all(r[3] for r in rows) for rd, rows in res.items()}

## fig2a-vis-cv-by-condition

> In visual cortex, median CV is higher in active behavior and blank screens (VISp 0.59 and 0.55, VISl 0.56 and 0.46, VISal 0.55 and 0.46) than in spontaneous activity (0.29, 0.30, 0.30), with natural images intermediate (0.39, 0.46, 0.48).

In [ ]:
E2A = []
for r, vals in {"VISp": (0.59, 0.55, 0.29, 0.39), "VISl": (0.56, 0.46, 0.30, 0.46), "VISal": (0.55, 0.46, 0.30, 0.48)}.items():
    for c, v in zip(["active", "blanks", "spont", "images"], vals):
        E2A.append((f"{r} {c}", r, c, "value", v, 0.1 * v))
for ur in UNITS:
    use(ur)
    full = show_expected(check_expected(MED_CV, E2A))
    orders = {}
    for rd in W:
        m = MED_CV[rd]
        order = {r: bool(m.loc[r, "active"] > m.loc[r, "images"] > m.loc[r, "spont"] and m.loc[r, "blanks"] > m.loc[r, "spont"]) for r in ["VISp", "VISl", "VISal"]}
        orders[rd] = all(order.values())
        print(f"reading {rd}: order active > images > spont and blanks > spont: {order}")
    verdict("fig2a-vis-cv-by-condition", ur, "reproduced" if any(full.values()) else "partially_reproduced" if any(orders.values()) else "not_reproduced")
print(OUT["fig2a-vis-cv-by-condition"])

**Result: reproduced.** Under all four condition readings the twelve medians are within tolerance (reading a/a: VISp 0.615, 0.579, 0.299 and 0.399 against 0.59, 0.55, 0.29 and 0.39; VISl 0.577, 0.485, 0.303, 0.475; VISal 0.561, 0.483, 0.304, 0.484), and the order active > images > spontaneous with blanks > spontaneous holds in the three areas. 'all' also gives 12 of 12 (a/a and b/a); 'qc' gives 7 or 8 of 12. The outcome rests on the adopted unit reading ('vbn'); the table at the end gives the criterion under 'all' and 'qc'.

## fig2a-thalamus-cv-by-condition

> Thalamic populations show the same pattern with smaller effects, LP median CV 0.37 in active behavior against 0.28 in spontaneous activity, LGd 0.32 against 0.23.

In [ ]:
E2T = [("LP active", "LP", "active", "value", 0.37, 0.037), ("LP spont", "LP", "spont", "value", 0.28, 0.028),
       ("LGd active", "LGd", "active", "value", 0.32, 0.032), ("LGd spont", "LGd", "spont", "value", 0.23, 0.023)]
for ur in UNITS:
    use(ur)
    full = show_expected(check_expected(MED_CV, E2T))
    red = {}
    for rd in W:
        d = (MED_CV[rd]["active"] - MED_CV[rd]["spont"])
        red[rd] = bool(max(d['LP'], d['LGd']) < min(d['VISp'], d['VISl'], d['VISal']))
        print(f"reading {rd}: active minus spontaneous LP {d['LP']:.3f}, LGd {d['LGd']:.3f}; VISp {d['VISp']:.3f}, VISl {d['VISl']:.3f}, VISal {d['VISal']:.3f}; "
              f"smaller in both thalamic nuclei than in all three visual areas: {red[rd]}")
    verdict("fig2a-thalamus-cv-by-condition", ur, "reproduced" if any(full[rd] and red[rd] for rd in W) else "not_reproduced")
print(OUT["fig2a-thalamus-cv-by-condition"])

**Result: reproduced.** Under every reading the four medians are within tolerance (LP 0.384 and 0.277 to 0.282, LGd 0.328 and 0.225 to 0.226 against 0.37, 0.28, 0.32 and 0.23), and the active minus spontaneous difference of LP (0.102 to 0.107) and LGd (0.102 to 0.103) is below that of VISp, VISl and VISal (0.256 to 0.316). 'all' gives the same; under 'qc' LP active (0.414) misses. The outcome rests on the adopted unit reading ('vbn'); the table at the end gives the criterion under 'all' and 'qc'.

## fig2a-hippocampus-cv-stable

> DG and CA3 keep nearly constant median CV across the visually driven conditions (DG 0.48 to 0.50, CA3 0.41 to 0.42; CA1 0.25), with modest increases in spontaneous activity (DG 0.51, CA3 0.43).

In [ ]:
E2H = [("DG VD low", "DG", VD, "min", 0.48, 0.048), ("DG VD high", "DG", VD, "max", 0.50, 0.05),
       ("CA3 VD low", "CA3", VD, "min", 0.41, 0.041), ("CA3 VD high", "CA3", VD, "max", 0.42, 0.042),
       ("CA1 VD each", "CA1", VD, "each", 0.25, 0.025),
       ("DG spont", "DG", "spont", "value", 0.51, 0.051), ("CA3 spont", "CA3", "spont", "value", 0.43, 0.043)]
for ur in UNITS:
    use(ur)
    full = show_expected(check_expected(MED_CV, E2H))
    up = {}
    for rd in W:
        m = MED_CV[rd]
        up[rd] = bool(m.loc['DG', 'spont'] > m.loc['DG', VD].max() and m.loc['CA3', 'spont'] > m.loc['CA3', VD].max())
        print(f"reading {rd}: spontaneous above every visually driven median: DG {bool(m.loc['DG','spont'] > m.loc['DG', VD].max())}, CA3 {bool(m.loc['CA3','spont'] > m.loc['CA3', VD].max())}")
    verdict("fig2a-hippocampus-cv-stable", ur, "reproduced" if any(full[rd] and up[rd] for rd in W) else "not_reproduced")
print(OUT["fig2a-hippocampus-cv-stable"])

**Result: reproduced.** Under every condition reading all seven values are within tolerance: DG 0.482 to 0.504 and CA3 0.406 to 0.415 across the visually driven conditions, CA1 0.254 to 0.258, DG spontaneous 0.509 to 0.510 and CA3 spontaneous 0.424 to 0.429, and the spontaneous medians of DG and CA3 exceed their visually driven ones. 'qc' also gives 7 of 7; under 'all' DG is higher (0.555 to 0.580) and misses. The outcome rests on the adopted unit reading ('vbn'); the table at the end gives the criterion under 'all' and 'qc'.

## fig2a-midbrain-cv-invariant

> Midbrain populations have essentially constant CV across all conditions (APN median 0.16 in every condition, MRN 0.20 to 0.21).

In [ ]:
E2M = [("APN each", "APN", paula.CONDITIONS, "each", 0.16, 0.016), ("MRN low", "MRN", paula.CONDITIONS, "min", 0.20, 0.02),
       ("MRN high", "MRN", paula.CONDITIONS, "max", 0.21, 0.021)]
for ur in UNITS:
    use(ur)
    full = show_expected(check_expected(MED_CV, E2M))
    print("   MRN medians, reading a/a:", MED_CV["a/a"].loc["MRN"].round(4).to_dict())
    verdict("fig2a-midbrain-cv-invariant", ur, "reproduced" if any(full.values()) else "not_reproduced")
print(OUT["fig2a-midbrain-cv-invariant"])

**Result: not reproduced.** The MRN highest median (0.209, natural images) is within 0.21 +/- 0.021, but APN's spontaneous median (0.142) is below 0.16 - 0.016 (its other conditions 0.164 to 0.165), and MRN's lowest (0.171, spontaneous) below 0.18. Under 'all' APN and the MRN low end match and the MRN high end (0.233) misses; under 'qc' all three miss. The outcome rests on the adopted unit reading ('vbn'); the table at the end gives the criterion under 'all' and 'qc'.

## JS divergence matrices of CV (claims of Fig. 2b)

For each condition, the JS divergence between the CV distributions of every pair of subregions, with 50 and 100 equal histogram bins over the pooled range of each pair, in bits and, as the claims file's variant, in natural logs (the bits value times ln 2). The sums are within each major area: (all) every pair of its block; (adjacent) the first off-diagonal in Table I order. Percentage differences, which do not depend on the log base: (second) |A - B| / B; (mean) |A - B| over the mean of A and B.

In [ ]:
LN2 = np.log(2)
def js_all(Wd, value, regs):
    """Keys (condition reading, bins, log base): base 2 computed, natural log as bits x ln 2."""
    out = {}
    for rd, w in Wd.items():
        for nb in (50, 100):
            m2 = {c: paula.js_matrix(w[w.condition == c], value, regs, nb) for c in paula.CONDITIONS}
            out[(rd, nb, "2")] = m2
            out[(rd, nb, "e")] = {c: m * LN2 for c, m in m2.items()}
    return out

def pct_table(JS):
    rows = []
    for (rd, nb, base), mats in JS.items():
        if base != "2":
            continue  # percentage differences do not depend on the log base
        for sm in ("all", "adjacent"):
            for pm in ("second", "mean"):
                for a in paula.AREAS:
                    s = {c: paula.summed_js(mats[c], a, sm) for c in paula.CONDITIONS}
                    for c1, c2 in paula.COND_PAIRS:
                        rows.append({"reading": rd, "bins": nb, "sum": sm, "pct": pm, "area": a, "pair": f"{c1}-{c2}",
                                     "S1": s[c1], "S2": s[c2], "diff": paula.pct_diff(s[c1], s[c2], pm)})
    return pd.DataFrame(rows)

for ur in UNITS:
    CACHE[ur]["JS_CV"] = js_all(WD[ur], "cv", REGS[ur])
    CACHE[ur]["PCT_CV"] = pct_table(CACHE[ur]["JS_CV"])
P = CACHE[MAIN]["PCT_CV"]
print(P[(P.reading == "a/a") & (P.bins == 50) & (P.pct == "second")].pivot_table(index=["sum", "area"], columns="pair", values="diff").round(1).to_string())
print("summed JS (units=vbn, reading a/a, 50 bins, all pairs, bits):", {a: {c: round(paula.summed_js(CACHE[MAIN]["JS_CV"][("a/a", 50, "2")][c], a, "all"), 2) for c in paula.CONDITIONS} for a in paula.AREAS})

## fig2b-vis-summed-js-cv

> For CV, the summed within-area JS divergences of visual cortex are nearly equal in active behavior and blank screens (about 1% apart) and differ by about 30% from those of natural images and spontaneous activity.

In [ ]:
FOUR = ["images-active", "active-spont", "images-blanks", "blanks-spont"]
for ur in UNITS:
    use(ur)
    res2b = []
    for key, g in PCT_CV[PCT_CV.area == "VIS"].groupby(["reading", "bins", "sum", "pct"]):
        d = g.set_index("pair")["diff"]
        ab = d["active-blanks"]
        four = d[FOUR]
        full = 0.5 <= ab <= 1.5 and bool(((four >= 27) & (four <= 33)).all())
        part = d.idxmin() == "active-blanks" and bool((d.drop("active-blanks") > 15).all())
        res2b.append({"reading": key[0], "bins": key[1], "sum": key[2], "pct": key[3], "active-blanks": round(ab, 2),
                      **{p: round(four[p], 1) for p in FOUR}, "closest pair": d.idxmin(), "reproduced": full, "partial": part})
    res2b = pd.DataFrame(res2b)
    print(res2b.to_string())
    print("combinations reproduced:", int(res2b.reproduced.sum()), "| partial:", int(res2b.partial.sum()), "of", len(res2b))
    verdict("fig2b-vis-summed-js-cv", ur, "reproduced" if res2b.reproduced.any() else "partially_reproduced" if res2b.partial.any() else "not_reproduced")
print(OUT["fig2b-vis-summed-js-cv"])

**Result: partially reproduced.** No combination of readings meets the full rule. With 100 bins, the sum over all within-area pairs and the percentage over the mean of the two sums (4 of 32 combinations, one per condition reading), active and blank screens are the closest pair (8.46% to 9.05% apart) and the other pairs differ by 19.1% to 43.7%, which meets the partial rule; active against blanks is not within 1 +/- 0.5% in any combination (3.12% to 11.24%). 'all' also meets the partial rule (4 combinations, active against blanks 0.32% to 0.53%); 'qc' meets neither. The outcome depends on the histogram and sum readings. The outcome rests on the adopted unit reading ('vbn'); the table at the end gives the criterion under 'all' and 'qc'.

## fig2b-hp-mb-summed-js-cv

> For CV, the largest hippocampal percentage difference in summed JS divergence is between active behavior and spontaneous activity (about 15%), and midbrain has the smallest differences (at most 7%).

In [ ]:
for ur in UNITS:
    use(ur)
    res2h = []
    for key, g in PCT_CV.groupby(["reading", "bins", "sum", "pct"]):
        t = g.pivot_table(index="area", columns="pair", values="diff")
        hp = t.loc["HP"]
        mb_max = t.loc["MB"].max()
        mb_small = int((t.idxmin(axis=0) == "MB").sum())
        ok = tol(hp["active-spont"], 15, 1.5) and hp.idxmax() == "active-spont" and mb_max <= 7.7 and mb_small >= 4
        res2h.append({"reading": key[0], "bins": key[1], "sum": key[2], "pct": key[3], "HP active-spont": round(hp["active-spont"], 1),
                      "HP largest pair": hp.idxmax(), "MB max": round(mb_max, 1), "pairs where MB smallest": mb_small, "reproduced": ok})
    res2h = pd.DataFrame(res2h)
    print(res2h.to_string())
    print("combinations reproduced:", int(res2h.reproduced.sum()), "of", len(res2h))
    verdict("fig2b-hp-mb-summed-js-cv", ur, "reproduced" if res2h.reproduced.any() else "not_reproduced")
print(OUT["fig2b-hp-mb-summed-js-cv"])

**Result: not reproduced.** Active against spontaneous is the largest hippocampal difference under every combination, as the paper says, but it is 44.8% to 77.6%, not 15% +/- 1.5%, and the largest midbrain difference is 27.2% to 48.5%, not at most 7.7%; the midbrain has the smallest difference in at most 4 of the 6 pairs. Every unit reading gives the same outcome.

## fig2b-visually-driven-js-cv-low

> Among the three visually driven conditions, the CV-based JS divergences of cortical regions differ little between conditions (typically less than 4%).

In [ ]:
VDP = ["images-active", "images-blanks", "active-blanks"]
for ur in UNITS:
    use(ur)
    r = PCT_CV[(PCT_CV.area == "VIS") & PCT_CV.pair.isin(VDP)].pivot_table(index=["reading", "bins", "sum", "pct"], columns="pair", values="diff")
    r["n at most 4.4%"] = (r[VDP] <= 4.4).sum(axis=1)
    r["reproduced"] = r["n at most 4.4%"] >= 2
    print(r.round(1).to_string())
    print("combinations reproduced (percentage reading):", int(r.reproduced.sum()), "of", len(r))
    vi = np.triu_indices(len(VISR), 1)
    js_pass = []
    for (rd, nb, base), mats in JS_CV.items():
        fr = {c: float((mats[c].loc[VISR, VISR].values[vi] <= 0.22).mean()) for c in VD}
        js_pass.append(all(v > 0.5 for v in fr.values()))
        print(f"JS reading, {rd}, {nb} bins, log base {base}: fraction of VIS-VIS pairs at most 0.22: {({k: round(v, 2) for k, v in fr.items()})}; "
              f"VIS-VIS JS range {min(mats[c].loc[VISR, VISR].values[vi].min() for c in VD):.3f} to {max(mats[c].loc[VISR, VISR].values[vi].max() for c in VD):.3f}; passes {js_pass[-1]}")
    verdict("fig2b-visually-driven-js-cv-low", ur, "reproduced" if r.reproduced.any() else "partially_reproduced" if any(js_pass) else "not_reproduced")
print(OUT["fig2b-visually-driven-js-cv-low"])

**Result: partially reproduced.** Under the statement's percentage form no combination has two of the three pairs at or below 4.4% (active against blanks 3.1% to 11.2%, the pairs with natural images 18.7% to 49.9%). Under the legend's form, most VIS-VIS divergences are at most 0.22 in each visually driven condition (fractions 0.73 to 1.0) under every condition, histogram and log-base reading. The finding holds as the legend states it and not as the text states it, which is partially reproduced. Every unit reading gives the same outcome.

## fig2b-dg-ca3-high-js-cv

> DG and CA3 have high CV-based JS divergences in all four conditions, largest in spontaneous activity (median 0.84 for both) and blank screens (median 0.63 for both).

In [ ]:
for ur in UNITS:
    use(ur)
    passes = []
    for variant, others in [("all other subregions", None), ("other hippocampal subregions", HPR)]:
        for (rd, nb, base), mats in JS_CV.items():
            med = {}
            for reg in ["DG", "CA3"]:
                for c in paula.CONDITIONS:
                    row = mats[c].loc[reg].drop(reg)
                    if others:
                        row = row[[o for o in others if o != reg]]
                    med[(reg, c)] = row.median()
            ok = (tol(med[("DG", "spont")], 0.84, 0.084) and tol(med[("CA3", "spont")], 0.84, 0.084) and tol(med[("DG", "blanks")], 0.63, 0.063)
                  and tol(med[("CA3", "blanks")], 0.63, 0.063)
                  and all(med[(g, "spont")] == max(med[(g, c)] for c in paula.CONDITIONS) for g in ["DG", "CA3"]))
            passes.append(ok)
            print(f"{variant}, {rd}, {nb} bins, base {base}: " + ", ".join(f"{g} {c} {v:.3f}" for (g, c), v in med.items()) + f"; passes {ok}")
    verdict("fig2b-dg-ca3-high-js-cv", ur, "reproduced" if any(passes) else "not_reproduced")
print(OUT["fig2b-dg-ca3-high-js-cv"])

**Result: not reproduced.** Against the other 25 subregions (bits, 50 bins, a/a) DG's medians are 0.713 (spontaneous) and 0.345 (blanks) and CA3's 0.443 and 0.216, against 0.84 and 0.63; over all readings DG spontaneous is 0.700 to 0.774 in bits, and the spontaneous median is the largest of the four conditions for both. Against the other hippocampal subregions only, DG spontaneous is 0.736 to 0.791 (within 0.84 +/- 0.084 with 100 bins, 0.785 to 0.791), but CA3 (0.555 to 0.622) and the blank medians (DG 0.529 to 0.545, CA3 0.333 to 0.356) miss. Natural logs give lower values still. Every unit reading gives the same outcome.

## fig2b-legend-js-cv

> In the CV-based JS divergences, visual and thalamic regions diverge strongly between spontaneous and visually driven states (above 0.40 for most visual areas, above 0.30 for thalamic regions), DG and CA3 moderately (0.15 to 0.35), and midbrain least (below 0.15 for most condition pairs).

Reading (a): per subregion, the JS divergence between its own CV distributions in two conditions. Reading (b): per subregion, the mean absolute difference between its row of the region-pair matrix in spontaneous activity and in a visually driven condition (this notebook's operational form of "comparing a region's row"). A subregion's value for "spontaneous against visually driven" is the mean over the three visually driven conditions.

In [ ]:
def legend_values(rd, nb, base):
    w = W[rd]
    out = {}
    for reg in REG:
        x = w[w.region == reg]
        out[reg] = {f"{c1}-{c2}": paula.js_divergence(x[x.condition == c1].cv, x[x.condition == c2].cv, nb, base=2.0 if base == "2" else np.e)
                    for c1, c2 in paula.COND_PAIRS}
    a = pd.DataFrame(out).T
    mats = JS_CV[(rd, nb, base)]
    b = pd.DataFrame({reg: {f"{c1}-{c2}": float((mats[c1].loc[reg].drop(reg) - mats[c2].loc[reg].drop(reg)).abs().mean()) for c1, c2 in paula.COND_PAIRS} for reg in REG}).T
    return a, b

SVD = ["images-spont", "active-spont", "blanks-spont"]
for ur in UNITS:
    use(ur)
    res_leg = []
    for rd in W:
        for nb in (50, 100):
            for base in ("2", "e"):
                for lab, t in zip(["a", "b"], legend_values(rd, nb, base)):
                    sv = t[SVD].mean(axis=1)
                    dgca3 = t.loc[["DG", "CA3"]].values.ravel()
                    res_leg.append({"reading": rd, "bins": nb, "base": base, "measure": lab,
                                    "VIS > 0.44": f"{int((sv[VISR] > 0.44).sum())}/{len(VISR)}", "TH > 0.33": f"{int((sv[THR] > 0.33).sum())}/{len(THR)}",
                                    "DG, CA3 range": f"{dgca3.min():.3f}-{dgca3.max():.3f}", "MB < 0.165": f"{int((t.loc[MBR].values < 0.165).sum())}/{t.loc[MBR].size}",
                                    "passes": bool((sv[VISR] > 0.44).mean() > 0.5 and (sv[THR] > 0.33).mean() > 0.5 and within(dgca3, 0.135, 0.385).all() and (t.loc[MBR].values < 0.165).mean() > 0.5)})
                    if rd == "a/a" and nb == 50 and base == "2" and ur == MAIN:
                        print(f"measure {lab}, reading a/a, 50 bins:\n", t.round(3).T.to_string())
    res_leg = pd.DataFrame(res_leg)
    print(res_leg.to_string())
    verdict("fig2b-legend-js-cv", ur, "reproduced" if res_leg.passes.any() else "not_reproduced")
print(OUT["fig2b-legend-js-cv"])

**Result: not reproduced.** Under reading a, 3 of 6 visual subregions exceed 0.44 in bits (none in natural logs), at most 2 of 9 thalamic subregions exceed 0.33, and the DG and CA3 values (0.002 to 0.153) lie mostly below 0.135; under reading b no visual or thalamic subregion passes (DG and CA3 0.024 to 0.418). Most midbrain values are below 0.165 under most readings (13 to 26 of 30). Every unit reading gives the same outcome.

## fig3a-reference-systems

> In the C-H plane with D = 6, white noise lies at H near 1 and C near 0, a sine near H 0 and C near 0.1, exponentially correlated Brownian motion at C 0.3 to 0.5 for H 0.3 to 0.6, and chaotic maps (logistic, tent, Ikeda) near maximum complexity, C 0.4 to 0.5 at H 0.5 to 0.8.

Synthetic series as defined in Supplemental Material Sec. I, 10^5 points after 10^3 dropped (claims file). Correlated Gaussian noise: increments from N(0, 1) noise convolved with the normalized kernel w_k = exp(-k / (K alpha)), K = floor(100 alpha) + 1, then summed (Sec. I.A). Near 1 is read as at least 0.9 and near 0 for C as at most 0.05 (10% of 0 is no tolerance). A sine cannot reach H = 0 with D = 6: it gives at least the rising and the falling pattern, so H >= ln 2 / ln 720 = 0.105; "H near 0" is read as at most that floor plus 0.05. The sine's sampling is not given, so 100, 1000 and 10 000 points per period are run. The logistic and tent maps are judged at their fully chaotic parameters (r = 4, mu = 1.9999, the tent map at mu = 2 collapsing to 0 in floating point). The Ikeda map's parameter p enters only through cos and sin of p - 6/(1 + x^2 + y^2), so p repeats every 2 pi; it is judged at the usual chaotic value 0.4 shifted into the supplement's [5, 17] (0.4 + 2 pi = 6.683 and 0.4 + 4 pi = 12.966), with a sweep shown.

In [ ]:
rng3 = np.random.default_rng(zlib.crc32(b"paula-2026/fig3a-reference-systems"))
N3, DROP = 100_000, 1_000
def logistic(r, n=N3 + DROP, x=0.4):
    out = np.empty(n)
    for i in range(n):
        x = r * x * (1 - x); out[i] = x
    return out[DROP:]
def tent(mu, n=N3 + DROP, x=0.3141):
    out = np.empty(n)
    for i in range(n):
        x = mu * x if x < 0.5 else mu * (1 - x); out[i] = x
    return out[DROP:]
def ikeda(p, u=0.9, n=N3 + DROP):
    x = y = 0.1; out = np.empty(n)
    for i in range(n):
        t = p - 6 / (1 + x * x + y * y)
        x, y = 1 + u * (x * np.cos(t) - y * np.sin(t)), u * (x * np.sin(t) + y * np.cos(t)); out[i] = x
    return out[DROP:]
def corr_walk(alpha, n=N3):
    K = int(np.floor(100 * alpha)) + 1
    w = np.exp(-np.arange(K) / (K * alpha)); w /= w.sum()
    e = rng3.standard_normal(n + K)
    return np.cumsum(np.convolve(e, w, mode="valid")[:n])
def fk_noise(k, n=N3):
    f = np.fft.rfftfreq(n); amp = np.zeros_like(f); amp[1:] = f[1:] ** (-k / 2)
    ph = np.exp(2j * np.pi * rng3.random(len(f)))
    return np.fft.irfft(amp * ph, n)
IK_CHAOS = [0.4 + 2 * np.pi, 0.4 + 4 * np.pi]  # the usual chaotic Ikeda parameter 0.4, shifted by the map's period 2 pi into [5, 17]
SINE_PP = [100, 1000, 10_000]
ref = []
ref.append(("white noise", "", *paula.bandt_pompe(rng3.standard_normal(N3))))
for pp in SINE_PP:
    ref.append(("sine", f"{pp} points per period", *paula.bandt_pompe(np.sin(2 * np.pi * np.arange(N3) / pp))))
ref.append(("Brownian motion", "", *paula.bandt_pompe(np.cumsum(rng3.standard_normal(N3)))))
for a in [0.01, 0.05, 0.1, 0.2, 0.3, 0.5, 0.7, 1.0]:
    ref.append(("exp. correlated walk", f"alpha={a}", *paula.bandt_pompe(corr_walk(a))))
for r in [3.5, 3.6, 3.7, 3.8, 3.9, 4.0]:
    ref.append(("logistic", f"r={r}", *paula.bandt_pompe(logistic(r))))
for mu in [1.2, 1.4, 1.6, 1.8, 1.9999]:
    ref.append(("tent", f"mu={mu}", *paula.bandt_pompe(tent(mu))))
for p in [5, 6, 6.5, IK_CHAOS[0], 7, 8, 11, IK_CHAOS[1], 14, 17]:
    ref.append(("Ikeda", f"p={p:.3f}", *paula.bandt_pompe(ikeda(p))))
for k in [0, 0.5, 1, 1.5, 2, 2.5, 3]:
    ref.append(("1/f^k noise", f"k={k}", *paula.bandt_pompe(fk_noise(k))))
REF = pd.DataFrame(ref, columns=["system", "parameter", "H", "C"])
print(REF.round(4).to_string())
g = lambda s, p: REF[(REF.system == s) & (REF.parameter == p)].iloc[0]
H_FLOOR = np.log(2) / np.log(720)  # a sine gives at least two ordinal patterns (rising and falling)
wn = g("white noise", "")
ok_wn = wn.H >= 0.9 and wn.C <= 0.05
print(f"white noise H {wn.H:.3f} C {wn.C:.4f}: {ok_wn}")
ok_sn = {}
for pp in SINE_PP:
    sn = g("sine", f"{pp} points per period")
    ok_sn[pp] = bool(sn.H <= H_FLOOR + 0.05 and within(sn.C, 0.09, 0.11))
    print(f"sine, {pp} points per period: H {sn.H:.4f} (floor {H_FLOOR:.4f}; near 0 read as at most floor + 0.05 = {H_FLOOR + 0.05:.3f}), C {sn.C:.4f} (0.09 to 0.11): {ok_sn[pp]}")
walks = REF[REF.system == "exp. correlated walk"]
ew = walks[within(walks.H, 0.27, 0.66)]
ok_ew = len(ew) > 0 and bool(within(ew.C, 0.27, 0.55).all())
print(f"exp. correlated walk: {len(ew)} of 8 alphas with H in 0.27 to 0.66, their C {ew.C.min():.3f} to {ew.C.max():.3f}, inside 0.27 to 0.55 {int(within(ew.C, 0.27, 0.55).sum())}; "
      f"all walks H {walks.H.min():.3f} to {walks.H.max():.3f}: {ok_ew}")
chaos = pd.DataFrame([g("logistic", "r=4.0"), g("tent", "mu=1.9999"), g("Ikeda", f"p={IK_CHAOS[0]:.3f}"), g("Ikeda", f"p={IK_CHAOS[1]:.3f}")])
chaos["inside"] = within(chaos.C, 0.36, 0.55) & within(chaos.H, 0.45, 0.88)
ok_ch = bool(chaos.inside.all())
print("chaotic maps at chaotic parameters (Ikeda at 0.4 + 2 pi and 0.4 + 4 pi):\n", chaos.round(3).to_string(), "\n inside C 0.36 to 0.55 and H 0.45 to 0.88:", ok_ch)
ik = REF[REF.system == "Ikeda"]
print("Ikeda sweep: inside the chaotic range at", list(ik.parameter[within(ik.C, 0.36, 0.55) & within(ik.H, 0.45, 0.88)]))
print("parts that hold: white noise", ok_wn, "| sine at some sampling", any(ok_sn.values()), "| correlated walk", ok_ew, "| chaotic maps", ok_ch)
OUT["fig3a-reference-systems"] = {"synthetic": "reproduced" if ok_wn and any(ok_sn.values()) and ok_ew and ok_ch else
                                  "partially_reproduced" if (ok_wn or ok_ch) else "not_reproduced"}
fig, ax = plt.subplots(figsize=(6, 4.5))
for s, gg in REF.groupby("system"):
    ax.plot(gg.H, gg.C, "o-" if len(gg) > 1 else "o", label=s, ms=4)
ax.set_xlabel("H"); ax.set_ylabel("C"); ax.set_xlim(0, 1); ax.set_ylim(0, 0.55); ax.legend(fontsize=7); plt.show()
print("plotted H range", REF.H.min().round(3), REF.H.max().round(3), "C range", REF.C.min().round(3), REF.C.max().round(3))
print(OUT["fig3a-reference-systems"])

**Result: partially reproduced.** White noise is at H 0.999, C 0.001, and the logistic map (r = 4) and tent map (mu = 1.9999) are at H 0.63, C 0.48, inside the chaotic range. The sine meets its reading at 10 000 points per period (H 0.107, C 0.106) and not at 100 or 1000 (C 0.165 and 0.113). The Ikeda map at its chaotic parameter (p = 6.683 and 12.966) lies at H 0.81, C 0.355, just below C 0.36; of the sweep only p = 6.5 is inside (H 0.794, C 0.367). The exponentially correlated walks with H in 0.27 to 0.66 have C 0.233 to 0.334, two of four below 0.27. The parameters of Fig. 3a are not in the text, so these positions rest on the parameter readings shown.

## fig3b-complexity-levels-by-region

> Across the hour of active recording, visual cortex and hippocampus keep the highest statistical complexity (C about 0.13 to 0.15), thalamus is intermediate (about 0.11) and midbrain lowest (about 0.10), and these differences stay stable over time.

Reading (a): the 36 active windows; reading (b): the eight natural-image windows per population in image order (alphabetical image names, as no other order is given).

In [ ]:
def fig3b(w, label):
    tc = w.groupby(["area", "k"]).C.agg(["mean", "sem", "size"]).reset_index()
    piv = tc.pivot(index="k", columns="area", values="mean")
    means = piv.mean()
    ok_v = all(within(means[a], 0.117, 0.165) for a in ["VIS", "HP"])
    ok_t, ok_m = tol(means["TH"], 0.11, 0.011), tol(means["MB"], 0.10, 0.01)
    order = (piv.MB < piv.TH) & (piv.TH < piv.VIS) & (piv.TH < piv.HP)
    print(f"{label}: mean over windows {means.round(4).to_dict()}; VIS and HP in 0.117 to 0.165 {ok_v}; TH {ok_t}; MB {ok_m}; "
          f"order MB < TH < VIS and HP at {int(order.sum())} of {len(order)} indices; range over indices {piv.min().round(3).to_dict()} to {piv.max().round(3).to_dict()}; "
          f"largest standard error {tc['sem'].max():.4f}")
    return piv, ok_v and ok_t and ok_m and bool(order.all()), ok_v and ok_t and ok_m and order.mean() > 0.5
for ur in UNITS:
    use(ur)
    piv_a, full_a, part_a = fig3b(ACT, "reading a (active)")
    piv_b, full_b, part_b = fig3b(IMG, "reading b (images)")
    verdict("fig3b-complexity-levels-by-region", ur, "reproduced" if full_a or full_b else "partially_reproduced" if part_a or part_b else "not_reproduced")
    if ur == MAIN:
        fig, ax = plt.subplots(figsize=(7, 3.5))
        for a in paula.AREAS:
            ax.plot(piv_a.index, piv_a[a], label=a)
        ax.set_xlabel("active window index"); ax.set_ylabel("mean C"); ax.legend(); plt.show()
print(OUT["fig3b-complexity-levels-by-region"])

**Result: reproduced.** Under reading a (the 36 active windows), the mean C over windows is 0.158 (VIS), 0.159 (HP), 0.108 (TH) and 0.098 (MB), each within its tolerance, and the order MB < TH < VIS and HP holds at all 36 window indices. Under reading b (natural images) hippocampus falls below thalamus (0.107 against 0.118) and the order holds at none of the 8 indices. Every unit reading gives the same outcome; it rests on reading a.

## C-H plane per area (claims of Figs. 3c to 3f)

Active windows (the reading the claims file assigns from the point counts), with natural-image windows shown. Spans are the 2.5th to 97.5th percentiles (and the 0th to 100th, a variant), compared unrounded. The Mann-Whitney test compares C of the windows with C of their surrogates, two-sided, under the five seeds; as windows of one population are not independent, the same comparison is also made on one median per population (Wilcoxon signed-rank).

In [ ]:
SPAN = {"VIS": ((0.8, 1.0), (0.1, 0.4)), "TH": ((0.75, 0.95), (0.05, 0.25)), "HP": ((0.85, 1.0), (0.05, 0.4)), "MB": ((0.9, 1.0), (0.05, 0.2))}
NPTS = {"VIS": 21084, "TH": 11265, "HP": 13059, "MB": 6296}
def ch_table(w, sfxs, surr=True):
    """Spans unrounded (rounded only for printing); Mann-Whitney on windows, and Wilcoxon on per-population medians."""
    rows = []
    for a in paula.AREAS:
        x = w[w.area == a]
        r = {"area": a, "windows": len(x), "paper windows": NPTS[a]}
        for q, (lo, hi) in [("95", (2.5, 97.5)), ("100", (0, 100))]:
            r[f"H span {q}"] = tuple(np.percentile(x.H, [lo, hi]))
            r[f"C span {q}"] = tuple(np.percentile(x.C, [lo, hi]))
        if surr:
            r["surrogate median C"] = x.s_C.median()
            r["empirical median C"] = x.C.median()
            for s in sfxs:
                r["p" + s] = stats.mannwhitneyu(x.C, x["s_C" + s]).pvalue
                r["surr med" + s] = x["s_C" + s].median()
            pm = x.groupby(["session", "region"])[["C", "s_C"]].median()
            r["p populations (Wilcoxon)"] = stats.wilcoxon(pm.C, pm.s_C).pvalue
            r["populations"] = len(pm)
        rows.append(r)
    return pd.DataFrame(rows).set_index("area")
def show_ch(t):
    return t.apply(lambda col: col.map(lambda v: tuple(round(float(u), 3) for u in v) if isinstance(v, tuple) else v)).T.to_string()
for ur in UNITS:
    use(ur)
    CACHE[ur]["CH"] = ch_table(ACT, SFXU)
    print(show_ch(CACHE[ur]["CH"]))
    if ur == MAIN:
        print("\nnatural-image windows:\n", show_ch(ch_table(IMG, SFXU, surr=False)))

def judge_ch(a, q="95"):
    (hl, hh), (cl, ch) = SPAN[a]
    H0, H1 = CH.loc[a, f"H span {q}"]; C0, C1 = CH.loc[a, f"C span {q}"]
    hw, cw = widen(hl, hh), widen(cl, ch)
    span_ok = hw[0] <= H0 and H1 <= hw[1] and cw[0] <= C0 and C1 <= cw[1]
    test_ok = CH.loc[a, "p"] < 0.001 and CH.loc[a, "empirical median C"] > CH.loc[a, "surrogate median C"]
    width = {b: CH.loc[b, f"C span {q}"][1] - CH.loc[b, f"C span {q}"][0] for b in paula.AREAS}
    print(f"{a}, span {q}: H {H0:.4f}-{H1:.4f} within {hw[0]:.3f}-{hw[1]:.3f}, C {C0:.4f}-{C1:.4f} within {cw[0]:.3f}-{cw[1]:.3f}: {span_ok}; "
          f"Mann-Whitney p {CH.loc[a, 'p']:.1e} (seeds run: {len(SFXU)}, largest {max(CH.loc[a, 'p' + s] for s in SFXU):.1e}), empirical above surrogate {CH.loc[a, 'empirical median C'] > CH.loc[a, 'surrogate median C']}; "
          f"per-population Wilcoxon p {CH.loc[a, 'p populations (Wilcoxon)']:.1e}; C span widths {({b: round(v, 3) for b, v in width.items()})}")
    return span_ok, test_ok, width

## fig3c-vis-ch-plane

> Visual cortex windows span H 0.8 to 1.0 and C 0.1 to 0.4 in the C-H plane, and differ from ISI-shuffled surrogates, which cluster at lower complexity (C 0.1 to 0.2), Mann-Whitney p < 0.001.

In [ ]:
for ur in UNITS:
    use(ur)
    res = {}
    for q in ["95", "100"]:
        res[q] = judge_ch("VIS", q)
    sm = [CH.loc["VIS", "surr med" + s] for s in SFXU]
    sm_ok = all(0.09 <= v <= 0.22 for v in sm)
    print(f"VIS surrogate median C over the seeds run {min(sm):.4f} to {max(sm):.4f}; inside 0.09 to 0.22: {sm_ok}")
    verdict("fig3c-vis-ch-plane", ur, "reproduced" if res["95"][0] and res["95"][1] and sm_ok else "not_reproduced")
print(OUT["fig3c-vis-ch-plane"])

**Result: not reproduced.** The test holds: C of the visual windows is above that of their surrogates (median 0.149 against 0.097, Mann-Whitney p = 0 at printed precision under all five seeds, per-population Wilcoxon p = 1.9e-141), and the surrogate median (0.0965 to 0.0966) lies in 0.09 to 0.22. The 2.5th to 97.5th percentile span of H (0.857 to 0.965) is inside 0.72 to 1.10, but that of C starts at 0.0887, below 0.090, a narrow miss; the 0th to 100th span misses on H (0.454). Under 'qc' the C span starts above 0.090 and the claim passes; 'all' misses as 'vbn' does. The outcome rests on the adopted unit reading ('vbn'); the table at the end gives the criterion under 'all' and 'qc'.

## fig3d-th-ch-plane

> Thalamic windows occupy a narrower part of the C-H plane than visual cortex (H 0.75 to 0.95, C 0.05 to 0.25), and differ from surrogates (p < 0.001).

In [ ]:
for ur in UNITS:
    use(ur)
    for q in ["95", "100"]:
        s_ok, t_ok, wd = judge_ch("TH", q)
        print(f"  span {q}: TH C span narrower than VIS: {wd['TH'] < wd['VIS']}")
        if q == "95":
            verdict("fig3d-th-ch-plane", ur, "reproduced" if s_ok and t_ok and wd["TH"] < wd["VIS"] else "not_reproduced")
print(OUT["fig3d-th-ch-plane"])

**Result: reproduced.** The thalamic span (H 0.929 to 0.969, C 0.081 to 0.167, 2.5th to 97.5th percentiles) is inside the widened ranges, its C span (0.087) is narrower than that of visual cortex (0.194), and C is above the surrogate's (Mann-Whitney p = 0 at printed precision, all five seeds; per-population Wilcoxon p = 1.1e-43). The 0th to 100th span misses on C (up to 0.344). Every unit reading gives the same outcome; it rests on the percentile reading of a span.

## fig3e-hp-ch-plane

> Hippocampal windows have the broadest complexity distribution (C 0.05 to 0.4) at high entropy (H 0.85 to 1.0), and differ from surrogates (p < 0.001).

In [ ]:
for ur in UNITS:
    use(ur)
    for q in ["95", "100"]:
        s_ok, t_ok, wd = judge_ch("HP", q)
        print(f"  span {q}: HP C span widest of the four: {max(wd, key=wd.get) == 'HP'}")
        if q == "95":
            verdict("fig3e-hp-ch-plane", ur, "reproduced" if s_ok and t_ok and max(wd, key=wd.get) == "HP" else "not_reproduced")
print(OUT["fig3e-hp-ch-plane"])

**Result: reproduced.** The hippocampal span (H 0.858 to 0.967, C 0.086 to 0.298) is inside the widened ranges, its C span (0.213) is the widest of the four areas, and C is above the surrogate's (p = 0 at printed precision, all five seeds; per-population Wilcoxon p = 7.7e-86). The 0th to 100th span misses on H (0.353) and C (0.441, against 0.440). Every unit reading gives the same outcome; it rests on the percentile reading of a span.

## fig3f-mb-ch-plane

> Midbrain windows occupy the most restricted part of the C-H plane, at high entropy (H 0.9 to 1.0) and low complexity (C 0.05 to 0.2), yet differ from surrogates (p < 0.001).

In [ ]:
for ur in UNITS:
    use(ur)
    for q in ["95", "100"]:
        s_ok, t_ok, wd = judge_ch("MB", q)
        print(f"  span {q}: MB C span narrowest of the four: {min(wd, key=wd.get) == 'MB'}")
        if q == "95":
            verdict("fig3f-mb-ch-plane", ur, "reproduced" if s_ok and t_ok and min(wd, key=wd.get) == "MB" else "not_reproduced")
    if ur == MAIN:
        fig, ax = plt.subplots(1, 4, figsize=(16, 3.5), sharey=True)
        for a_, a in zip(ax, paula.AREAS):
            x = ACT[ACT.area == a]
            a_.plot(x.H, x.C, ".", ms=1, alpha=0.3); a_.plot(x.s_H, x.s_C, ".", ms=1, alpha=0.3, color="k")
            a_.set_title(a); a_.set_xlim(0.5, 1); a_.set_ylim(0, 0.5); a_.set_xlabel("H")
        ax[0].set_ylabel("C"); plt.show()
        print("windows outside the axes (H < 0.5):", int((ACT.H < 0.5).sum()), "| C > 0.5:", int((ACT.C > 0.5).sum()))
print(OUT["fig3f-mb-ch-plane"])

**Result: reproduced.** The midbrain span (H 0.938 to 0.971, C 0.075 to 0.150) is inside the widened ranges, its C span (0.074) is the narrowest of the four, and C is above the surrogate's (Mann-Whitney p = 2.1e-166, at most 8.4e-161 over five seeds; per-population Wilcoxon p = 8.9e-23). Under the 0th to 100th span H starts at 0.808, below 0.81, and the C span is not the narrowest. Every unit reading gives the same outcome; it rests on the percentile reading of a span.

## fig3-complexity-range-fractions

> The fractions of windows by complexity range are, for visual cortex, 0.8% in [0.3, 0.4] and 4.9% in [0.0, 0.1]; thalamus 0.6% in [0.2, 0.3] and 24.6% in [0.0, 0.1]; hippocampus 2.7% in [0.3, 0.4] and 9.5% in [0.0, 0.1]; midbrain 45.4% in [0.1, 0.2] and 54.0% in [0.0, 0.1].

In [ ]:
FR = [("VIS", 0.3, 0.4, 0.8), ("VIS", 0.0, 0.1, 4.9), ("TH", 0.2, 0.3, 0.6), ("TH", 0.0, 0.1, 24.6),
      ("HP", 0.3, 0.4, 2.7), ("HP", 0.0, 0.1, 9.5), ("MB", 0.1, 0.2, 45.4), ("MB", 0.0, 0.1, 54.0)]
for ur in UNITS:
    use(ur)
    best = 0
    for lab, w in [("reading a (active)", ACT), ("reading b (images)", IMG)]:
        out, n = [], 0
        for a, lo, hi, v in FR:
            c = w[w.area == a].C
            f = 100 * ((c >= lo) & ((c < hi) | ((hi == 0.4) & (c <= hi)))).mean()
            ok = tol(f, v, 0.1 * v); n += ok
            out.append(f"{a} [{lo},{hi}) {f:.2f}% ({v})" + ("" if ok else " MISS"))
        best = max(best, n)
        print(f"{lab}: {n}/8 within tolerance; " + "; ".join(out))
    verdict("fig3-complexity-range-fractions", ur, "reproduced" if best == 8 else "partially_reproduced" if best >= 6 else "not_reproduced")
print(OUT["fig3-complexity-range-fractions"])

**Result: not reproduced.** None of the eight fractions is within tolerance on the active windows (for example VIS 10.45% in [0, 0.1) against 4.9%, TH 43.17% against 24.6%, MB 32.03% in [0.1, 0.2) against 45.4%), and none on the natural-image windows. Every unit reading gives the same outcome.

## Medians of C by condition (claims of Fig. 4a to 4d)

The medians of C are computed with those of CV, above, for each unit reading.

In [ ]:
print("median C, units=vbn:\n", CACHE[MAIN]["MED_C"].round(3).to_string())

def mw_pairs(w, regs, bonf):
    rows = []
    for r in regs:
        x = w[w.region == r]
        for c1, c2 in paula.COND_PAIRS:
            p = stats.mannwhitneyu(x[x.condition == c1].C, x[x.condition == c2].C).pvalue
            rows.append({"region": r, "pair": f"{c1}-{c2}", "p": min(1.0, p * bonf)})
    return pd.DataFrame(rows)

## fig4a-vis-complexity-by-condition

> All six visual areas have higher median complexity during natural image presentation (0.16 to 0.26) than during spontaneous activity (0.1), with active behavior and blank screens intermediate (0.1 to 0.19), all pairwise comparisons p < 0.001.

In [ ]:
LEG4A = {"VISl": (0.15, 0.26), "VISal": (0.15, 0.25), "VISp": (0.16, 0.19)}
for ur in UNITS:
    use(ur)
    full, ordr = False, False
    for rd in W:
        m = MED_C[rd].loc[VISR]
        order = (m.images > m.active) & (m.images > m.blanks) & (m.active > m.spont) & (m.blanks > m.spont)
        ordr |= bool(order.all())
        ni = within(m.images, *widen(0.16, 0.26)); sp = within(m.spont, 0.09, 0.11); ab = within(m.active, *widen(0.1, 0.19)) & within(m.blanks, *widen(0.1, 0.19))
        for name, bonf in [("uncorrected", 1), ("Bonferroni 36", 36)]:
            mw = mw_pairs(W[rd], VISR, bonf)
            allp = bool((mw.p < 0.001).all())
            ok = bool(order.all() and ni.all() and sp.all() and ab.all() and allp)
            full |= ok
            print(f"reading {rd}, {name}: order holds in {int(order.sum())}/6 ({list(order[~order].index)} fail); images medians {m.images.min():.3f}-{m.images.max():.3f} inside {int(ni.sum())}/6; "
                  f"spont {m.spont.min():.3f}-{m.spont.max():.3f} inside {int(sp.sum())}/6; active and blanks {min(m.active.min(), m.blanks.min()):.3f}-{max(m.active.max(), m.blanks.max()):.3f} inside {int(ab.sum())}/6; "
                  f"tests p < 0.001 {int((mw.p < 0.001).sum())}/36, largest p {mw.p.max():.1e} ({mw.loc[mw.p.idxmax(), 'region']} {mw.loc[mw.p.idxmax(), 'pair']}); passes {ok}")
    verdict("fig4a-vis-complexity-by-condition", ur, "reproduced" if full else "partially_reproduced" if ordr else "not_reproduced")
    m = MED_C["a/a"].loc[VISR]
    print("medians, reading a/a:\n", m.round(3).to_string())
    iq = W["a/a"][W["a/a"].region.isin(VISR)].groupby(["region", "condition"]).C.agg(lambda x: x.quantile(0.75) - x.quantile(0.25)).unstack()
    print("IQRs (legend 0.08 to 0.12 in visually driven states):\n", iq.round(3).to_string())
    for r, (lo, hi) in LEG4A.items():
        vd = m.loc[r, VD]
        print(f"legend {r} {lo}-{hi}: visually driven medians {vd.min():.3f}-{vd.max():.3f}, inside widened range {bool(within(vd, *widen(lo, hi)).all())}")
print(OUT["fig4a-vis-complexity-by-condition"])

**Result: not reproduced.** All medians fall in the quoted ranges (natural images 0.158 to 0.258, spontaneous 0.094 to 0.098, active and blanks 0.102 to 0.187), and the legend's narrower ranges for VISl, VISal and VISp are met, but the order fails in VISp, where the natural-image median (0.174) is below the active one (0.187) and close to the blank one (0.175; Mann-Whitney p = 0.77 to 0.83); the other 35 of 36 pairwise tests give p < 0.001. The partial rule needs the order, so the claim is not reproduced. Every unit reading gives the same outcome.

## fig4b-thalamus-lgd-lp-modulated

> Of nine thalamic nuclei, only LGd and LP show significant modulation of complexity across conditions (p < 0.001), higher during visually driven states (0.14 to 0.18) than spontaneous activity (0.11 to 0.12), with smaller effects than visual cortex.

The effect is the visually driven minus spontaneous median, taken (i) as the largest visually driven median minus the spontaneous one and (ii) as the mean of the three visually driven medians minus the spontaneous one; the claims file does not fix it, and both are shown.

In [ ]:
def thal_tests(w, regs):
    rows = []
    for r in regs:
        x = w[w.region == r]
        kw = stats.kruskal(*[x[x.condition == c].C for c in paula.CONDITIONS]).pvalue
        row = {"region": r, "kruskal p": kw}
        for c1, c2 in paula.COND_PAIRS:
            row[f"{c1}-{c2}"] = stats.mannwhitneyu(x[x.condition == c1].C, x[x.condition == c2].C).pvalue
        pw = [row[f"{c}-spont"] for c in VD]
        row["largest VD-spont p"] = max(pw)
        row["smallest VD-spont p"] = min(pw)
        row["smallest of six p"] = min(row[f"{c1}-{c2}"] for c1, c2 in paula.COND_PAIRS)
        rows.append(row)
    return pd.DataFrame(rows).set_index("region")
for ur in UNITS:
    use(ur)
    passes = {"i": False, "ii": False}
    for rd in W:
        m = MED_C[rd]
        tt = thal_tests(W[rd], THR)
        eff_max = m.loc[:, VD].max(axis=1) - m.spont
        eff_mean = m.loc[:, VD].mean(axis=1) - m.spont
        res = []
        for r in ["LGd", "LP"]:
            vd = m.loc[r, VD]
            text_ok = bool(within(vd, *widen(0.14, 0.18)).all()) and within(m.loc[r, "spont"], *widen(0.11, 0.12))
            leg = {"LP": ((0.12, 0.14), 0.11), "LGd": ((0.12, 0.15), 0.10)}[r]
            leg_ok = bool(within(vd, *widen(*leg[0])).all()) and tol(m.loc[r, "spont"], leg[1], 0.1 * leg[1])
            test_ok = tt.loc[r, "kruskal p"] < 0.001 or tt.loc[r, "largest VD-spont p"] < 0.001
            res.append((r, vd.round(3).to_dict(), round(m.loc[r, "spont"], 3), text_ok, leg_ok, bool((vd > m.loc[r, "spont"]).all()),
                        bool(eff_max[r] < eff_max[VISR].min()), bool(eff_mean[r] < eff_mean[VISR].min()), test_ok))
        for form, k in [("i", 6), ("ii", 7)]:
            passes[form] |= all(x[8] and x[5] and (x[3] or x[4]) and x[k] for x in res)
        print(f"reading {rd}:\n", tt[["kruskal p", "largest VD-spont p"]].to_string(float_format=lambda v: f"{v:.1e}"))
        for x in res:
            print(f"  {x[0]}: VD medians {x[1]}, spont {x[2]}; text values {x[3]}; legend values {x[4]}; VD above spont {x[5]}; effect below every visual area (i) {x[6]}, (ii) {x[7]}")
        print(f"  effects (i) VIS {eff_max[VISR].round(3).to_dict()} TH {eff_max[THR].round(3).to_dict()}")
        print(f"  effects (ii) VIS {eff_mean[VISR].round(3).to_dict()} TH {eff_mean[THR].round(3).to_dict()}")
    print(f"passes with effect form (i): {passes['i']}; with form (ii): {passes['ii']}")
    verdict("fig4b-thalamus-lgd-lp-modulated", ur, "reproduced" if passes["i"] else "not_reproduced")
print(OUT["fig4b-thalamus-lgd-lp-modulated"])

**Result: not reproduced.** LGd and LP are modulated under both test readings (Kruskal-Wallis p at most 6.4e-66 and 3.0e-64; every visually driven condition against spontaneous p at most 3.2e-21 and 2.9e-19), and their visually driven medians (0.112 to 0.149) exceed the spontaneous ones (0.094 to 0.098). LGd matches the legend's values, but LP's spontaneous median (0.098 under every condition reading) falls just below the legend's 0.11 - 0.011 = 0.099, and neither nucleus matches the text's (LGd active 0.121 and LP active 0.123 below 0.126), so the value part fails. The effects of LGd (0.054 to 0.056) and LP (0.044 to 0.045) are below every visual area's under form (i) (0.062 to 0.163); under form (ii) LGd (0.032 to 0.034) is above VISrl (0.028 to 0.030). Under 'all' and 'qc' LP's spontaneous median is within the legend's range and the claim is reproduced, so the outcome turns on the unit reading. The outcome rests on the adopted unit reading ('vbn'); the table at the end gives the criterion under 'all' and 'qc'.

## fig4b-other-thalamus-not-modulated

> The other seven thalamic nuclei (MGd, PoT, SGN, POL, MGm, MGv, PIL) keep uniformly low complexity (0.10 to 0.13) with no significant stimulus dependence.

Test readings: Kruskal-Wallis across the four conditions, and the pairwise Mann-Whitney tests of each visually driven condition against spontaneous activity. For this claim of absence a nucleus counts as modulated under the pairwise reading when any of its three pairs reaches p < 0.001; the lenient form (all three pairs) is shown too. All six pairwise p-values of each nucleus are printed.

In [ ]:
SEVEN = ["MGd", "PoT", "SGN", "POL", "MGm", "MGv", "PIL"]
for ur in UNITS:
    use(ur)
    seven = [r for r in SEVEN if r in REG]
    ok_any = {"Kruskal-Wallis": False, "pairwise, any VD-spont pair": False, "pairwise, all three VD-spont pairs": False}
    for rd in W:
        m = MED_C[rd].loc[seven]
        tt = thal_tests(W[rd], seven)
        rng_ok = bool(within(m.values, 0.09, 0.143).all())
        mod = {"Kruskal-Wallis": list(tt.index[tt["kruskal p"] < 0.001]),
               "pairwise, any VD-spont pair": list(tt.index[tt["smallest VD-spont p"] < 0.001]),
               "pairwise, all three VD-spont pairs": list(tt.index[tt["largest VD-spont p"] < 0.001])}
        for k, v in mod.items():
            ok_any[k] |= rng_ok and not v and len(seven) == 7
        print(f"reading {rd}: medians {m.values.min():.3f}-{m.values.max():.3f} inside 0.09 to 0.143 {rng_ok}; nuclei found modulated (p < 0.001): {mod}")
        print("   largest condition difference in median C:", (m.max(axis=1) - m.min(axis=1)).round(4).to_dict())
        print("   ", tt.to_string(float_format=lambda v: f"{v:.1e}").replace("\n", "\n    "))
    print("reproduced under:", ok_any)
    strict = [ok_any["Kruskal-Wallis"], ok_any["pairwise, any VD-spont pair"]]
    verdict("fig4b-other-thalamus-not-modulated", ur, "reproduced" if all(strict) else "not_reproduced" if not any(strict) else "test readings differ")
print(OUT["fig4b-other-thalamus-not-modulated"])

**Result: not reproduced.** The medians of the seven nuclei are 0.092 to 0.107, inside 0.09 to 0.143, and their largest condition difference is at most 0.0055, but both test readings find modulation: Kruskal-Wallis gives p < 0.001 in MGd, SGN, MGm and MGv (5.7e-08 to 7.3e-04), and the pairwise reading (a nucleus is modulated when any visually driven condition differs from spontaneous at p < 0.001, the reading suited to a claim of absence) finds MGd, SGN and MGv (smallest p 1.6e-07, MGv images against spontaneous). Only the lenient form of the pairwise reading, which counts a nucleus as modulated when all three conditions differ, finds none (or SGN alone under b/a). Every unit reading gives the same outcome.

## fig4c-dg-ca3-complexity

> DG and CA3 show elevated complexity across most conditions (DG 0.19 to 0.27, CA3 0.15 to 0.21), peaking during spontaneous activity, opposite to visual cortex, while CA1, POST, ProS and SUB stay lower and stable (0.11 to 0.15).

In [ ]:
for ur in UNITS:
    use(ur)
    passes = {"text": False, "legend": False}
    peak_any = False
    for rd in W:
        m = MED_C[rd]
        # (a) text: DG and CA3 inside their range in at least three of four conditions; CA1, POST, ProS, SUB inside 0.11-0.15 in every condition
        a_dg = int(within(m.loc["DG"], *widen(0.19, 0.27)).sum()); a_ca3 = int(within(m.loc["CA3"], *widen(0.15, 0.21)).sum())
        rest = m.loc[["CA1", "POST", "ProS", "SUB"]]
        a_rest = bool(within(rest.values, *widen(0.11, 0.15)).all())
        peak = all(m.loc[g].idxmax() == "spont" for g in ["DG", "CA3"])
        # (b) legend: natural images apart
        b_ok = (tol(m.loc["DG", "images"], 0.12, 0.012) and tol(m.loc["CA3", "images"], 0.11, 0.011) and tol(m.loc["CA1", "images"], 0.10, 0.010)
                and bool(within(m.loc["DG", ["active", "blanks", "spont"]], *widen(0.19, 0.26)).all())
                and bool(within(m.loc["CA3", ["active", "blanks", "spont"]], *widen(0.13, 0.18)).all())
                and bool(within(m.loc["CA1", ["active", "blanks", "spont"]], *widen(0.12, 0.17)).all())
                and bool(within(m.loc[["POST", "ProS", "SUB"]].values, *widen(0.10, 0.13)).all()))
        passes["text"] |= a_dg >= 3 and a_ca3 >= 3 and a_rest and peak
        passes["legend"] |= b_ok and peak
        print(f"reading {rd}: DG {m.loc['DG'].round(4).to_dict()}; CA3 {m.loc['CA3'].round(4).to_dict()}; spontaneous peak {peak}")
        print(f"   text: DG inside in {a_dg}/4, CA3 in {a_ca3}/4, CA1/POST/ProS/SUB all inside {a_rest} (range {rest.values.min():.3f}-{rest.values.max():.3f}); passes {a_dg >= 3 and a_ca3 >= 3 and a_rest and peak}")
        print(f"   legend: passes {b_ok and peak}; CA1 {m.loc['CA1'].round(3).to_dict()}; POST/ProS/SUB {m.loc[['POST','ProS','SUB']].values.min():.3f}-{m.loc[['POST','ProS','SUB']].values.max():.3f}")
        peak_any |= peak
    verdict("fig4c-dg-ca3-complexity", ur, "reproduced" if any(passes.values()) else "partially_reproduced" if peak_any else "not_reproduced")
print(OUT["fig4c-dg-ca3-complexity"])

**Result: reproduced.** Under the legend's reading, with every condition reading, DG natural images 0.116, CA3 0.107 and CA1 0.095 match the legend's 0.12, 0.11 and 0.10, DG (0.179 to 0.251), CA3 (0.130 to 0.188) and CA1 (0.118 to 0.158) match their elevated ranges in the other conditions, POST, ProS and SUB lie in 0.096 to 0.135, and DG and CA3 peak in spontaneous activity. The text's reading fails: CA3 is inside its range in 2 of 4 conditions, and CA1 (0.095 in natural images) falls below 0.099. Every unit reading gives the same outcome; it rests on the legend reading.

## fig4d-midbrain-complexity-invariant

> Most midbrain areas (NB, MRN, SCig, APN) keep uniformly low complexity (0.10 to 0.12) across conditions; only NOT shows modest stimulus dependence (0.13 to 0.15 in visually driven states against 0.13 in spontaneous activity).

In [ ]:
LEG4D = {"NB": (0.11, 0.12), "MRN": (0.10, 0.11), "SCig": (0.10, 0.11), "APN": (0.09, 0.09)}
for ur in UNITS:
    use(ur)
    if "NOT" not in REG or any(r not in REG for r in LEG4D):
        print("not testable: subregions not kept under this reading:", [r for r in ["NOT"] + list(LEG4D) if r not in REG])
        verdict("fig4d-midbrain-complexity-invariant", ur, "not testable (NOT not kept)")
        continue
    passes = False
    for rd in W:
        m = MED_C[rd]
        four = m.loc[["NB", "MRN", "SCig", "APN"]]
        text_ok = bool(within(four.values, *widen(0.10, 0.12)).all())
        leg_ok = all(within(m.loc[r], *widen(*v)).all() for r, v in LEG4D.items())
        rngs = m.loc[MBR].max(axis=1) - m.loc[MBR].min(axis=1)
        not_largest = rngs.idxmax() == "NOT"
        passes |= (text_ok or leg_ok) and not_largest
        print(f"reading {rd}: medians\n{four.round(3).to_string()}\n NOT {m.loc['NOT'].round(3).to_dict()}; ranges across conditions {rngs.round(3).to_dict()}; "
              f"text values {text_ok}; legend values {leg_ok}; NOT range largest {not_largest}; passes text {text_ok and not_largest}, legend {leg_ok and not_largest}")
    verdict("fig4d-midbrain-complexity-invariant", ur, "reproduced" if passes else "not_reproduced")
print(OUT["fig4d-midbrain-complexity-invariant"])

**Result: reproduced.** Under the legend's values every condition median of NB (0.104 to 0.111), MRN (0.094 to 0.101), SCig (0.095 to 0.097) and APN (0.086 to 0.089) lies in its widened range, and NOT has the largest range across conditions (0.055 to 0.057, from 0.101 to 0.159, against at most 0.007). Under the text's values APN falls below 0.09. 'all' gives the same; 'qc' does not keep NOT, so the claim cannot be judged under it. The outcome rests on the legend reading. The outcome rests on the adopted unit reading ('vbn'); the table at the end gives the criterion under 'all' and 'qc'.

## JS divergence matrices of C (claims of Figs. 4e to 4h)

As for CV: per condition, JS divergence between the C distributions of every pair of subregions, with 50 and 100 histogram bins, in bits and (claims file, variant) in natural logs.

In [ ]:
for ur in UNITS:
    CACHE[ur]["JS_C"] = js_all(WD[ur], "C", REGS[ur])
def pair_vals(m, a_set, b_set):
    """Values of pairs with one subregion in a_set and the other in b_set (each unordered pair once)."""
    out = []
    for i, j in zip(*iu):
        ai, aj = AREA_OF[i], AREA_OF[j]
        if (ai in a_set and aj in b_set) or (aj in a_set and ai in b_set):
            out.append(m.values[i, j])
    return np.array(out)
use(MAIN)
print("JS of C (bits), units=vbn, reading a/a, 50 bins, mean by area pair:")
for c in paula.CONDITIONS:
    m = JS_C[("a/a", 50, "2")][c]
    print(c, {f"{a}-{b}": round(float(pair_vals(m, {a}, {b}).mean()), 3) for a in paula.AREAS for b in paula.AREAS if a <= b})

## fig4e-js-complexity-natural-images

> During natural image presentation the complexity distributions of visual areas are similar to each other (JS below 0.2) but diverge from thalamus and midbrain (JS above 0.4), and DG and CA3 diverge moderately from other regions (JS 0.3 to 0.5).

In [ ]:
for ur in UNITS:
    use(ur)
    passes = []
    for (rd, nb, base), mats in JS_C.items():
        m = mats["images"]
        vv = pair_vals(m, {"VIS"}, {"VIS"}); vt = pair_vals(m, {"VIS"}, {"TH"}); vm = pair_vals(m, {"VIS"}, {"MB"})
        dg = np.r_[m.loc["DG"].drop("DG").values, m.loc["CA3"].drop(["CA3", "DG"]).values]
        ok = (vv < 0.22).mean() > 0.5 and (vt > 0.36).mean() > 0.5 and (vm > 0.36).mean() > 0.5 and within(dg, 0.27, 0.55).mean() > 0.5
        passes.append(ok)
        print(f"{rd}, {nb} bins, base {base}: VIS-VIS below 0.22 {(vv < 0.22).mean():.2f} (range {vv.min():.3f}-{vv.max():.3f}); VIS-TH above 0.36 {(vt > 0.36).mean():.2f} (median {np.median(vt):.3f}); "
              f"VIS-MB above 0.36 {(vm > 0.36).mean():.2f} (median {np.median(vm):.3f}); DG/CA3 pairs in 0.27-0.55 {within(dg, 0.27, 0.55).mean():.2f} (median {np.median(dg):.3f}); passes {ok}")
    verdict("fig4e-js-complexity-natural-images", ur, "reproduced" if any(passes) else "not_reproduced")
print(OUT["fig4e-js-complexity-natural-images"])

**Result: not reproduced.** In natural images (bits) all or most VIS-VIS divergences are below 0.22 (0.80 to 1.00) and most VIS-TH (0.70 to 0.72) and VIS-MB (0.73 to 0.77) divergences exceed 0.36, but only 0.18 to 0.20 of the pairs of DG or CA3 with other subregions lie in 0.27 to 0.55 (median 0.199 to 0.217). In natural logs the VIS-TH and VIS-MB parts fail too (0.37 to 0.43). Every unit reading gives the same outcome.

## fig4fg-js-complexity-active-blanks

> During active behavior and blank screens visual cortex keeps complexity distributions distinct from subcortical structures (JS above 0.4), with within-hippocampus divergences of 0.2 to 0.4 during active behavior.

In [ ]:
for ur in UNITS:
    use(ur)
    passes = []
    for (rd, nb, base), mats in JS_C.items():
        f = {c: pair_vals(mats[c], {"VIS"}, {"TH", "HP", "MB"}) for c in ["active", "blanks"]}
        hh = pair_vals(mats["active"], {"HP"}, {"HP"})
        ok = all((v > 0.36).mean() > 0.5 for v in f.values()) and within(hh, 0.18, 0.44).mean() > 0.5
        passes.append(ok)
        print(f"{rd}, {nb} bins, base {base}: VIS-subcortical above 0.36, active {(f['active'] > 0.36).mean():.2f} (median {np.median(f['active']):.3f}), blanks {(f['blanks'] > 0.36).mean():.2f} (median {np.median(f['blanks']):.3f}); "
              f"HP-HP active in 0.18-0.44 {within(hh, 0.18, 0.44).mean():.2f} (range {hh.min():.3f}-{hh.max():.3f}); passes {ok}")
    verdict("fig4fg-js-complexity-active-blanks", ur, "reproduced" if any(passes) else "not_reproduced")
print(OUT["fig4fg-js-complexity-active-blanks"])

**Result: not reproduced.** Fewer than half of the pairs of a visual subregion with a subcortical one exceed 0.36 (0.38 in active behavior, 0.31 to 0.34 in blanks, in bits; medians 0.189 to 0.265), and 0.33 of the hippocampal pairs in active behavior lie in 0.18 to 0.44 (0.13 in natural logs). Every unit reading gives the same outcome.

## fig4h-js-complexity-spontaneous-converges

> In spontaneous activity the JS divergences between regions are lower overall, visual cortex differing less from other regions (JS 0.2 to 0.4) than in visually driven states, and midbrain divergences stay low in all conditions (below 0.2 within the midbrain group).

In [ ]:
for ur in UNITS:
    use(ur)
    passes = []
    for (rd, nb, base), mats in JS_C.items():
        allm = {c: mats[c].values[iu].mean() for c in paula.CONDITIONS}
        vn = {c: pair_vals(mats[c], {"VIS"}, {"TH", "HP", "MB"}) for c in paula.CONDITIONS}
        mm = {c: pair_vals(mats[c], {"MB"}, {"MB"}) for c in paula.CONDITIONS}
        c1 = all(allm["spont"] < allm[c] for c in VD)
        c2 = all(vn["spont"].mean() < vn[c].mean() for c in VD) and within(vn["spont"], 0.18, 0.44).mean() > 0.5
        c3 = all((mm[c] < 0.22).mean() > 0.5 for c in paula.CONDITIONS)
        passes.append(c1 and c2 and c3)
        print(f"{rd}, {nb} bins, base {base}: mean of all pairs {({c: round(v, 3) for c, v in allm.items()})} (spont lowest {c1}); "
              f"mean VIS-other {({c: round(float(v.mean()), 3) for c, v in vn.items()})}, spont pairs in 0.18-0.44 {within(vn['spont'], 0.18, 0.44).mean():.2f} ({c2}); "
              f"MB-MB below 0.22 {({c: round(float((v < 0.22).mean()), 2) for c, v in mm.items()})} ({c3}); passes {c1 and c2 and c3}")
    verdict("fig4h-js-complexity-spontaneous-converges", ur, "reproduced" if any(passes) else "not_reproduced")
print(OUT["fig4h-js-complexity-spontaneous-converges"])

**Result: not reproduced.** The mean visual-to-other divergence is lowest in spontaneous activity (bits: 0.215 to 0.265 against 0.274 to 0.518), as the statement says, but the mean of all pairs is not (0.282 to 0.353 in spontaneous against 0.226 to 0.322 in the others), only 0.26 to 0.40 of the spontaneous visual-to-other pairs lie in 0.18 to 0.44, and midbrain pairs are below 0.22 in all conditions only in some natural-log readings (in bits 0.2 to 0.3 of them in spontaneous activity). No reading meets all three parts. Every unit reading gives the same outcome.

## supp-robustness-parameters

> The hierarchical, stimulus-dependent pattern of complexity across regions holds for other parameter choices (bin sizes 20 and 50 ms, D from 3 to 6, tau from 1 to 4).

The visually driven minus spontaneous difference of an area is the median C of all its visually driven windows minus the median of its spontaneous windows (this notebook's operational form).

In [ ]:
print("supplement's 20 ms counts: images 11 321, active 51 452, blanks 11 265, spontaneous 2856")
for ur in UNITS:
    use(ur)
    w = W["a/a"]
    print("windows per condition (the 20 ms series use the same windows):", w.condition.value_counts().to_dict())
    nok = 0
    for bw, d, tau in paula.ROBUST + [(0.05, 6, 1)]:
        col = "C" if (bw, d, tau) == (0.05, 6, 1) else f"C_{int(bw*1000)}_{d}_{tau}"
        m = w.groupby(["region", "condition"])[col].median().unstack()
        vis_ok = bool((m.loc[VISR, "images"] > m.loc[VISR, "spont"]).all())
        diff = {a: w[(w.area == a) & w.condition.isin(VD)][col].median() - w[(w.area == a) & (w.condition == "spont")][col].median() for a in paula.AREAS}
        ok = vis_ok and diff["VIS"] > diff["MB"]
        if col != "C":
            nok += ok
        print(f"bin {int(bw*1000)} ms, D={d}, tau={tau}{' (main)' if col == 'C' else ''}: images above spontaneous in {int((m.loc[VISR, 'images'] > m.loc[VISR, 'spont']).sum())}/6 visual areas; "
              f"VD minus spont by area {({a: round(v, 4) for a, v in diff.items()})}; holds {ok}")
    print("combinations of the supplement where the pattern holds:", nok, "of 5")
    verdict("supp-robustness-parameters", ur, "reproduced" if nok == 5 else "partially_reproduced" if nok >= 3 else "not_reproduced")
print(OUT["supp-robustness-parameters"])

**Result: partially reproduced.** The pattern (natural images above spontaneous in every visual area, and a larger visually driven minus spontaneous difference in visual cortex than in midbrain) holds for 4 of the 5 combinations; with bin 20 ms, D = 3, tau = 4 natural images exceed spontaneous in 2 of 6 visual areas and the visual difference is -0.0002. Every unit reading gives the same outcome. It rests on this notebook's form of the area difference.

## Complexity against CV (claims of Fig. 5)

Per area and condition, windows binned by CV, the mean C and its standard error per bin. Readings: bin width 0.1 or 0.05, with no minimum count or a minimum of 10 windows per bin. Spans are the occupied bins (lower edge of the first to upper edge of the last) and the range of bin means. Reading a/a of the conditions; the condition readings change the spontaneous and blank windows only slightly (medians above). The table and figure are for 'vbn'.

In [ ]:
for ur in UNITS:
    TRAJ = {}
    w0 = WD[ur]["a/a"]
    for width in (0.1, 0.05):
        for mn in (0, 10):
            for a in paula.AREAS:
                for c in paula.CONDITIONS:
                    x = w0[(w0.area == a) & (w0.condition == c)]
                    t = paula.trajectory(x.cv, x.C, width, mn)
                    mono = bool((np.diff(t["mean"].values) >= -(t.se.values[1:] + t.se.values[:-1])).all())
                    rho = stats.spearmanr(x.cv, x.C).statistic
                    slope = stats.linregress(x.cv, x.C).slope
                    TRAJ[(width, mn, a, c)] = {"cv lo": t.lo.min(), "cv hi": t.hi.max(), "C lo": t["mean"].min(), "C hi": t["mean"].max(),
                                               "bins": len(t), "monotonic": mono, "spearman": rho, "slope": slope, "table": t}
    CACHE[ur]["TRAJ"] = TRAJ
    TR = pd.DataFrame({k: {kk: vv for kk, vv in v.items() if kk != "table"} for k, v in TRAJ.items()}).T
    TR.index.names = ["width", "min", "area", "condition"]
    if ur == MAIN:
        print(TR.astype({"cv lo": float, "cv hi": float, "C lo": float, "C hi": float, "spearman": float, "slope": float}).round(3).to_string())
use(MAIN)
fig, ax = plt.subplots(1, 4, figsize=(16, 3.5), sharey=True)
for a_, c in zip(ax, paula.CONDITIONS):
    for a in paula.AREAS:
        t = TRAJ[(0.1, 10, a, c)]["table"]
        a_.errorbar((t.lo + t.hi) / 2, t["mean"], yerr=t.se, label=a, marker="o", ms=3)
    a_.set_title(c); a_.set_xlabel("CV"); a_.set_xlim(0, 2)
ax[0].set_ylabel("mean C"); ax[0].legend(); plt.show()
print("bins with CV above the axis (2):", int(sum((TRAJ[(0.1, 10, a, c)]["table"].lo >= 2).sum() for a in paula.AREAS for c in paula.CONDITIONS)))

def span_ok(x, lo, hi):
    return tol(x[0], lo, 0.1 * abs(lo)) and tol(x[1], hi, 0.1 * abs(hi))

## fig5-vis-complexity-cv-coupling

> In visual cortex complexity rises with CV during visually driven conditions (natural images CV 0.1 to 0.9 and C 0.08 to 0.34; active behavior CV 0.2 to 1.6 and C 0.10 to 0.38; blanks CV 0.1 to 1.2 and C 0.10 to 0.39), while in spontaneous activity the trajectory compresses, with C below about 0.15.

In [ ]:
Q5 = {"images": ((0.1, 0.9), (0.08, 0.34)), "active": ((0.2, 1.6), (0.10, 0.38)), "blanks": ((0.1, 1.2), (0.10, 0.39))}
for ur in UNITS:
    use(ur)
    passes = []
    for width in (0.1, 0.05):
        for mn in (0, 10):
            oks = []
            for c, (cvr, cr) in Q5.items():
                r = TRAJ[(width, mn, "VIS", c)]
                s1, s2 = span_ok((r["cv lo"], r["cv hi"]), *cvr), span_ok((r["C lo"], r["C hi"]), *cr)
                rise = r["monotonic"] and r["spearman"] > 0
                oks.append(s1 and s2 and rise)
                print(f"width {width}, min {mn}, {c}: CV {r['cv lo']:.2f}-{r['cv hi']:.2f} (quoted {cvr}) {s1}; C {r['C lo']:.3f}-{r['C hi']:.3f} (quoted {cr}) {s2}; "
                      f"monotonic {r['monotonic']}, Spearman {r['spearman']:.3f}")
            sp = TRAJ[(width, mn, "VIS", "spont")]
            passes.append(all(oks) and sp['C hi'] < 0.165)
            print(f"width {width}, min {mn}: spontaneous maximum of bin means {sp['C hi']:.3f} (below 0.165: {sp['C hi'] < 0.165}); passes {passes[-1]}")
    verdict("fig5-vis-complexity-cv-coupling", ur, "reproduced" if any(passes) else "not_reproduced")
print(OUT["fig5-vis-complexity-cv-coupling"])

**Result: not reproduced.** Complexity rises with CV in visual cortex in the three visually driven conditions (Spearman 0.640, 0.844 and 0.836; monotonic under bins of 0.05 with at least 10 windows), but no reading matches all spans: the natural-image bins reach CV 1.00 to 1.40 against 0.9 +/- 0.09, and without a minimum count the spontaneous maximum of the bin means is 0.337 to 0.350; with at least 10 windows per bin the spontaneous maximum is 0.153, below 0.165, but the natural-image and blank spans miss. Every unit reading gives the same outcome.

## fig5-th-weaker-coupling

> Thalamus shows a weaker but consistent positive complexity-CV relationship in the visually driven conditions, with shallower slopes than visual cortex and a restricted range (CV 0.2 to 0.8, C 0.10 to 0.20).

In [ ]:
for ur in UNITS:
    use(ur)
    passes = []
    for width in (0.1, 0.05):
        for mn in (0, 10):
            oks = []
            for c in VD:
                r, v = TRAJ[(width, mn, "TH", c)], TRAJ[(width, mn, "VIS", c)]
                s = 0.18 <= r["cv lo"] and r["cv hi"] <= 0.88 and 0.09 <= r["C lo"] and r["C hi"] <= 0.22
                oks.append(r["slope"] > 0 and r["slope"] < v["slope"] and s)
                print(f"width {width}, min {mn}, {c}: TH slope {r['slope']:.4f} vs VIS {v['slope']:.4f}; TH CV {r['cv lo']:.2f}-{r['cv hi']:.2f}, C {r['C lo']:.3f}-{r['C hi']:.3f}; spans inside {s}")
            passes.append(all(oks))
            print(f"  passes {all(oks)}")
    verdict("fig5-th-weaker-coupling", ur, "reproduced" if any(passes) else "not_reproduced")
print(OUT["fig5-th-weaker-coupling"])

**Result: not reproduced.** In each visually driven condition the thalamic slope is positive and below the visual one (0.094 to 0.129 against 0.226 to 0.372), but the thalamic spans are wider than CV 0.18 to 0.88 and C 0.09 to 0.22 under every reading (for example CV from 0.10 or 0.15, and up to 1.05 in active behavior with bins of 0.05 and at least 10 windows). Every unit reading gives the same outcome.

## fig5-hp-spontaneous-elevated

> Hippocampus shows modest complexity (C 0.12 to 0.19) in the visually driven conditions but markedly elevated complexity during spontaneous activity (C 0.12 to 0.25) at intermediate CV (0.4 to 0.7), comparable to visual cortex during image viewing.

In [ ]:
for ur in UNITS:
    use(ur)
    passes = []
    for width in (0.1, 0.05):
        for mn in (0, 10):
            t = TRAJ[(width, mn, "HP", "spont")]["table"]
            mid = t[(t.lo >= 0.4 - 1e-9) & (t.hi <= 0.7 + 1e-9)]
            top = mid["mean"].max() if len(mid) else np.nan
            vdmax = {c: TRAJ[(width, mn, "HP", c)]["C hi"] for c in VD}
            ok = tol(top, 0.25, 0.025) and all(top > v for v in vdmax.values()) and all(tol(v, 0.19, 0.019) for v in vdmax.values())
            passes.append(ok)
            print(f"width {width}, min {mn}: HP spontaneous bin means at CV 0.4-0.7 up to {top:.3f}; HP visually driven maxima {({c: round(v, 3) for c, v in vdmax.items()})}; passes {ok}")
    verdict("fig5-hp-spontaneous-elevated", ur, "reproduced" if any(passes) else "not_reproduced")
print(OUT["fig5-hp-spontaneous-elevated"])

**Result: not reproduced.** The hippocampal spontaneous bin means at CV 0.4 to 0.7 reach 0.227 to 0.239, within 0.25 +/- 0.025, but they do not exceed the hippocampal maximum of every visually driven condition (active 0.198 to 0.441), and the visually driven maxima are not all within 0.19 +/- 0.019 (natural images 0.135 with at least 10 windows per bin, 0.336 without). Every unit reading gives the same outcome.

## fig5-mb-flat

> Midbrain complexity stays low (C 0.10 to 0.13) across the full range of CV (0.1 to 0.9) in all conditions, giving nearly flat trajectories.

In [ ]:
for ur in UNITS:
    use(ur)
    passes = []
    for width in (0.1, 0.05):
        for mn in (0, 10):
            inside = {c: bool(TRAJ[(width, mn, "MB", c)]["C lo"] >= 0.09 and TRAJ[(width, mn, "MB", c)]["C hi"] <= (0.165 if c == "images" else 0.143)) for c in paula.CONDITIONS}
            slopes = {c: {a: abs(TRAJ[(width, mn, a, c)]["slope"]) for a in paula.AREAS} for c in paula.CONDITIONS}
            flat = {c: min(s, key=s.get) == "MB" for c, s in slopes.items()}
            passes.append(all(inside.values()) and all(flat.values()))
            print(f"width {width}, min {mn}: MB bin means {({c: (round(TRAJ[(width, mn, 'MB', c)]['C lo'], 3), round(TRAJ[(width, mn, 'MB', c)]['C hi'], 3)) for c in paula.CONDITIONS})}; "
                  f"inside {inside}; MB slope smallest {flat}; |slopes| {({c: {a: round(v, 3) for a, v in s.items()} for c, s in slopes.items()})}; passes {passes[-1]}")
    verdict("fig5-mb-flat", ur, "reproduced" if any(passes) else "not_reproduced")
print(OUT["fig5-mb-flat"])

**Result: not reproduced.** The midbrain slope of C on CV is the smallest of the four areas in all four conditions, and its bin means lie in range in natural images and spontaneous activity, but in active behavior and blanks they start at 0.084, below 0.09, under every reading. Every unit reading gives the same outcome.

## supp-fig6-mean-correlation

> In visual cortex the mean pairwise spike count correlation is elevated during active behavior and blank screens (median 0.08 to 0.13) and near zero during natural images and spontaneous activity (0.00 to 0.01); midbrain values are low and invariant.

In [ ]:
for ur in UNITS:
    use(ur)
    passes = []
    for rd in W:
        for col in ["corr50", "corr20"]:
            m = W[rd].groupby(["region", "condition"])[col].median().unstack()
            v = m.loc[VISR]
            ab = v[["active", "blanks"]].values; ns = v[["images", "spont"]].values
            mb = m.loc[MBR]
            mbr = (mb.max(axis=1) - mb.min(axis=1))
            ok = bool(within(ab, *widen(0.08, 0.13)).all() and (ns < 0.011).all() and (mbr < 0.02).all())
            passes.append(ok)
            print(f"{rd}, {col}: VIS active/blanks {ab.min():.3f}-{ab.max():.3f} inside 0.072-0.143 {int(within(ab, *widen(0.08, 0.13)).sum())}/12; "
                  f"images/spont {ns.min():.4f}-{ns.max():.4f} below 0.011 {int((ns < 0.011).sum())}/12; MB range across conditions {mbr.round(4).to_dict()}; passes {ok}")
            if rd == "a/a" and col == "corr50" and ur == MAIN:
                print(m.loc[VISR + MBR].round(4).to_string())
    verdict("supp-fig6-mean-correlation", ur, "reproduced" if any(passes) else "not_reproduced")
print(OUT["supp-fig6-mean-correlation"])

**Result: not reproduced.** The visual medians in active behavior and blanks are 0.014 to 0.055 (50 ms) and 0.007 to 0.031 (20 ms), below 0.072, and the natural-image and spontaneous medians are 0.005 to 0.038, half or more of them above 0.011. Midbrain medians vary by less than 0.02 across conditions except NOT (0.023 to 0.024 at 50 ms). Every unit reading gives the same outcome.

## complexity-separates-images-from-spontaneous

> Neither CV nor mean pairwise correlation separates natural image presentation from spontaneous activity in visual cortex, while statistical complexity does.

In [ ]:
for ur in UNITS:
    use(ur)
    js_any, iqr_any = False, False
    for rd in W:
        w = W[rd]
        for nb in (50, 100):
            rows = []
            for r in VISR:
                x = w[w.region == r]
                a, b = x[x.condition == "images"], x[x.condition == "spont"]
                row = {"region": r}
                for col in ["C", "cv", "corr50"]:
                    row[f"JS {col}"] = paula.js_divergence(a[col], b[col], nb)
                    qa, qb = a[col].quantile([0.25, 0.75]).values, b[col].quantile([0.25, 0.75]).values
                    row[f"IQR overlap {col}"] = bool(qa[0] <= qb[1] and qb[0] <= qa[1])
                rows.append(row)
            t = pd.DataFrame(rows).set_index("region")
            js_ok = ((t["JS C"] > t["JS cv"]) & (t["JS C"] > t["JS corr50"])).sum()
            iqr_ok = ((~t["IQR overlap C"]) & t["IQR overlap cv"] & t["IQR overlap corr50"]).sum()
            js_any |= js_ok > 3
            iqr_any |= iqr_ok > 3
            print(f"{rd}, {nb} bins: complexity separates while CV and correlation do not, JS reading {js_ok}/6, IQR reading {iqr_ok}/6")
            if rd == "a/a" and nb == 50:
                print(t.round(3).to_string())
    verdict("complexity-separates-images-from-spontaneous", ur, "reproduced" if js_any or iqr_any else "not_reproduced")
print(OUT["complexity-separates-images-from-spontaneous"])

**Result: reproduced.** Under the JS reading, in all six visual subregions the divergence between natural images and spontaneous activity is larger for complexity (0.437 to 0.837, reading a/a, 50 bins) than for CV (0.249 to 0.537) and mean correlation (0.189 to 0.479), under every condition and histogram reading. Under the IQR reading it holds in 1 or 2 of 6, since the CV and correlation IQRs of the two conditions often do not overlap either. Every unit reading gives the same outcome; it rests on the JS reading.

## Outcome of each criterion under each unit reading

The verdicts computed in the claim sections, collected in one table, so that the dependence of each outcome on the unit reading can be seen.

In [ ]:
# outcome of each claim's criterion under each unit reading, as computed in the claim sections above
OUT["table1-sample"] = {ur: "not_reproduced" for ur in UNITS}  # 153 sessions under every reading (section table1-sample)
byur = pd.DataFrame(OUT).T.reindex(columns=UNITS + ["synthetic"])
print(byur.fillna("").to_string())
dep = byur[UNITS].dropna()
print("claims whose criterion gives a different result under the variant readings than under 'vbn':",
      list(dep.index[(dep["all"] != dep["vbn"]) | (dep["qc"] != dep["vbn"])]))

## Summary

| Claim | Outcome |
|---|---|
| table1-sample | not reproduced |
| fig1c-cv-region-medians | partially reproduced (assumed) |
| fig1c-cv-above-surrogate | partially reproduced (assumed) |
| fig1d-surrogate-abolishes-cv-correlations | not reproduced (assumed) |
| fig1e-positive-cv-modules | reproduced (assumed) |
| fig1e-dg-ca3-anticorrelation-hubs | reproduced (assumed) |
| fig1e-midbrain-weakest-cv-correlations | not reproduced (assumed) |
| fig2a-vis-cv-by-condition | reproduced (assumed) |
| fig2a-thalamus-cv-by-condition | reproduced (assumed) |
| fig2a-hippocampus-cv-stable | reproduced (assumed) |
| fig2a-midbrain-cv-invariant | not reproduced (assumed) |
| fig2b-vis-summed-js-cv | partially reproduced (assumed) |
| fig2b-hp-mb-summed-js-cv | not reproduced (assumed) |
| fig2b-visually-driven-js-cv-low | partially reproduced (assumed) |
| fig2b-dg-ca3-high-js-cv | not reproduced (assumed) |
| fig2b-legend-js-cv | not reproduced (assumed) |
| fig3a-reference-systems | partially reproduced (synthetic series, assumed) |
| fig3b-complexity-levels-by-region | reproduced (assumed) |
| fig3c-vis-ch-plane | not reproduced (assumed) |
| fig3d-th-ch-plane | reproduced (assumed) |
| fig3e-hp-ch-plane | reproduced (assumed) |
| fig3f-mb-ch-plane | reproduced (assumed) |
| fig3-complexity-range-fractions | not reproduced (assumed) |
| fig4a-vis-complexity-by-condition | not reproduced (assumed) |
| fig4b-thalamus-lgd-lp-modulated | not reproduced (assumed) |
| fig4b-other-thalamus-not-modulated | not reproduced (assumed) |
| fig4c-dg-ca3-complexity | reproduced (assumed) |
| fig4d-midbrain-complexity-invariant | reproduced (assumed) |
| fig4e-js-complexity-natural-images | not reproduced (assumed) |
| fig4fg-js-complexity-active-blanks | not reproduced (assumed) |
| fig4h-js-complexity-spontaneous-converges | not reproduced (assumed) |
| supp-robustness-parameters | partially reproduced (assumed) |
| fig5-vis-complexity-cv-coupling | not reproduced (assumed) |
| fig5-th-weaker-coupling | not reproduced (assumed) |
| fig5-hp-spontaneous-elevated | not reproduced (assumed) |
| fig5-mb-flat | not reproduced (assumed) |
| supp-fig6-mean-correlation | not reproduced (assumed) |
| complexity-separates-images-from-spontaneous | reproduced (assumed) |

"Assumed" marks an outcome that rests on a choice the paper does not make; the first of them, for every claim on the recordings, is the unit selection: the 'vbn' filter (snr > 1, ISI violations < 1, firing rate > 0.1 spikes/s, the filter of the Allen Institute's Visual Behavior Neuropixels tutorial), whose neuron counts per population match Table I within 5% in each area. The table before this summary gives each criterion under the claims file's two unit readings too; ten claims (fig1c-cv-region-medians, fig1e-positive-cv-modules, fig1e-dg-ca3-anticorrelation-hubs, fig2a-vis-cv-by-condition, fig2a-thalamus-cv-by-condition, fig2a-hippocampus-cv-stable, fig2b-vis-summed-js-cv, fig3c-vis-ch-plane, fig4b-thalamus-lgd-lp-modulated, fig4d-midbrain-complexity-invariant) come out differently under 'all' or 'qc'.

A failure to reproduce here may be a failure of this attempt, not of the paper: see the reasoning and the differences from the paper's method.

How the analysis differs from the paper's. The paper's 102 sessions are not identified, and every one of the 153 ecephys sessions of the dandiset meets both session readings, so all 153 are used; no grouping by genotype, image set, novelty or probe count gives 102, and the populations and windows are about 1.5 times the paper's. The unit filter is not stated; the 'vbn' filter is adopted because it matches Table I's neurons per population. The surrogate of each window is drawn once under the seed of `fig1c-cv-above-surrogate` (and four more) and serves every claim on surrogates. The JS histograms and log base, spans, the blank and spontaneous windows, the test of "modulation", the session combination of Fig. 1d and 1e and several effect sizes are readings named in the claims file or in the claim sections above, each shown. Tests on windows treat windows as independent, as the paper appears to; the C-H and surrogate comparisons are repeated on one median per population.

What was found about the dataset. The 153 sessions (81 mice) each hold an active block of 3599 to 3611 s, a 289-s gray screen after it and eight images; no recording appears twice (no shared spike-time fingerprint, spike count, or subject and start time). The visual cortex responds to every session's flash onsets and not to onsets shifted into the gray screen or to omitted flashes, so the spike times and the stimulus table share a clock in all 153 sessions. An image is shown 320 to 1056 times per session (median 572.5), so its blank series is about 286 s long (median), not the "approximately 240 s" of Methods II.A, which would need 480 flashes; leaving out change flashes gives 271 s. The location labels include generic ones (`TH`, `MB`, `HPF`) for units the atlas did not assign to a subregion. Part of the units carry the quality label 'noise' (the share is printed in the data check); the units table holds the metrics (snr, ISI violations, firing rate) that the Allen tutorial's filter uses.